# AI-Powered Network Intrusion Detection System

## Notebook 02 – Data Cleaning & Preparation

**Author:** Jithesh  
**Role:** Member A – ML & Data Engineer  
**Dataset:** CICIDS-2017  
**Date:** 09 August 2026

---

## Objective

This notebook performs data cleaning and preparation of the CICIDS-2017 dataset before feature engineering and model development.

### Tasks

- Validate and profile the available CICIDS-2017 dataset files
- Analyze target labels and dataset structure
- Identify missing, duplicate, invalid, and anomalous values
- Investigate the causes and context of detected data-quality issues
- Apply evidence-based cleaning and standardization decisions
- Consolidate and validate the cleaned dataset
- Prepare the dataset for feature engineering and model development

## Environment Setup

This section initializes the working environment by mounting Google Drive, importing the required libraries, defining the project paths, and verifying the available raw CICIDS-2017 dataset files.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### Import Libraries

In [2]:
import os
import pandas as pd
import numpy as np

### Define Project Paths

In [3]:
PROJECT_PATH = "/content/drive/MyDrive/AI-Network-Intrusion-Detection-System"

DATA_PATH = os.path.join(PROJECT_PATH, "data", "raw")

print(DATA_PATH)

/content/drive/MyDrive/AI-Network-Intrusion-Detection-System/data/raw


### Verify Raw Dataset Files

In [4]:
os.listdir(DATA_PATH)

['Wednesday-workingHours.pcap_ISCX.csv',
 'Tuesday-WorkingHours.pcap_ISCX.csv',
 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
 'Monday-WorkingHours.pcap_ISCX.csv',
 'Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv']

# Stage 1 – Dataset Consolidation & Validation

## Objective

The objective of this stage is to consolidate the available CICIDS-2017 network traffic CSV files and validate their structure before performing data cleaning.

The following aspects will be examined:

- Identify and load all available CICIDS-2017 CSV files
- Verify the structure and feature consistency across files
- Compare the number of records in each file
- Verify the target/label column and attack classes
- Identify any structural inconsistencies that may affect later preprocessing

The findings from this stage will determine the appropriate data cleaning and preparation steps.

### 1.1 Identify Available Dataset Files

The CICIDS-2017 dataset is distributed across multiple CSV files representing different days and attack scenarios.

Before loading the data, the available files in the raw dataset directory are listed to confirm the complete set of files available for processing.

In [5]:
# List all files in the raw dataset directory

raw_files = os.listdir(DATA_PATH)

for file in raw_files:
    print(file)

Wednesday-workingHours.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [6]:
# Count the number of CSV files

csv_files = [file for file in raw_files if file.lower().endswith(".csv")]

print("Number of CSV files:", len(csv_files))

Number of CSV files: 8


### 1.2 Create Dataset File Inventory

A file inventory is created to document the available CICIDS-2017 source files and their sizes before loading them into memory.

In [7]:
# Create an inventory of the raw CSV files

file_inventory = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    file_inventory.append({
        "File Name": file,
        "Size (MB)": round(os.path.getsize(file_path) / (1024 ** 2), 2)
    })

file_inventory_df = pd.DataFrame(file_inventory)

file_inventory_df

,File Name,Size (MB)
0,Wednesday-workingHours.pcap_ISCX.csv,214.74
1,Tuesday-WorkingHours.pcap_ISCX.csv,128.82
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,49.61
3,Thursday-WorkingHours-Afternoon-Infilteration....,79.25
4,Monday-WorkingHours.pcap_ISCX.csv,168.73
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,55.62
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,73.34
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,73.55


### 1.3 Load and Compare Dataset Files

The available CICIDS-2017 CSV files are loaded individually to examine their dimensions and verify that all files contain a compatible structure before they are consolidated.

Loading the files individually at this stage allows differences in the number of records, features, and dataset structure to be identified before combining them.

In [8]:
# Load each CSV file and record its basic structure

dataset_info = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    dataset_info.append({
        "File Name": file,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Label Column Present": " Label" in df.columns
    })

    del df

dataset_info_df = pd.DataFrame(dataset_info)

dataset_info_df

,File Name,Rows,Columns,Label Column Present
0,Wednesday-workingHours.pcap_ISCX.csv,692703,79,False
1,Tuesday-WorkingHours.pcap_ISCX.csv,445909,79,False
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,79,False
3,Thursday-WorkingHours-Afternoon-Infilteration....,288602,79,False
4,Monday-WorkingHours.pcap_ISCX.csv,529918,79,False
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,79,False
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,79,False
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,79,False


### 1.4 Compare Feature Schemas

The column structure of each dataset file is compared to determine whether the files share a consistent feature schema.

A common schema is required before the datasets can be safely consolidated into a unified CICIDS-2017 dataset.

In [ ]:
# Compare the column schemas of all CSV files

file_columns = {}

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path, nrows=5)
    df.columns = df.columns.str.strip()
    file_columns[file] = list(df.columns)

    del df

# Use the first file as the reference schema
reference_file = csv_files[0]
reference_columns = file_columns[reference_file]

schema_comparison = []

for file, columns in file_columns.items():
    schema_comparison.append({
        "File Name": file,
        "Column Count": len(columns),
        "Matches Reference": columns == reference_columns
    })

schema_comparison_df = pd.DataFrame(schema_comparison)

schema_comparison_df

,File Name,Column Count,Matches Reference
0,Wednesday-workingHours.pcap_ISCX.csv,79,True
1,Tuesday-WorkingHours.pcap_ISCX.csv,79,True
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,79,True
3,Thursday-WorkingHours-Afternoon-Infilteration....,79,True
4,Monday-WorkingHours.pcap_ISCX.csv,79,True
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,79,True
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,79,True
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,79,True


### Stage 1 Observations

The dataset inventory and structural validation confirmed that all eight CICIDS-2017 CSV files contain a consistent schema.

Key observations:

- Eight source CSV files were identified in the raw dataset directory.
- Each file contains 79 columns, including the `Label` target column.
- All files match the reference feature schema exactly.
- The files collectively contain approximately 2.83 million network-flow records.
- The consistent schema indicates that the files can be considered compatible for subsequent cleaning and consolidation.

The dataset structure is therefore suitable for proceeding to the data-quality assessment stage.

No records have been removed or modified during this stage.

## Stage 1 Conclusion

The CICIDS-2017 source files have been successfully inventoried and structurally validated. All available files share a common 79-feature schema and contain the required target label.

The next stage will focus on identifying and handling data-quality issues, particularly missing values, duplicate records, and anomalous values identified during dataset exploration.

# Stage 2 – Data Quality Assessment

### Objective

To identify data-quality issues across the complete CICIDS-2017 dataset before applying preprocessing and feature engineering.

This stage focuses on:

- Missing values
- Duplicate records
- Invalid or anomalous values
- Data-type consistency
- Target-label consistency

The findings from this stage will determine the cleaning operations required before the dataset is prepared for feature engineering and model development.

### 2.1 Missing Value Analysis

Missing values can affect statistical analysis, preprocessing, and model training. Each CICIDS-2017 source file is therefore checked independently to determine the number and proportion of missing values in each feature.

The analysis is performed file-wise to avoid unnecessarily loading the complete multi-million-row dataset into memory.

In [ ]:
# Analyze missing values across all dataset files

missing_value_summary = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    missing_count = df.isna().sum().sum()
    total_values = df.shape[0] * df.shape[1]

    missing_value_summary.append({
        "File Name": file,
        "Rows": df.shape[0],
        "Missing Values": missing_count,
        "Missing Percentage": round(
            (missing_count / total_values) * 100, 4
        )
    })

    del df

missing_value_df = pd.DataFrame(missing_value_summary)

missing_value_df

,File Name,Rows,Missing Values,Missing Percentage
0,Wednesday-workingHours.pcap_ISCX.csv,692703,1008,0.0018
1,Tuesday-WorkingHours.pcap_ISCX.csv,445909,201,0.0006
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,20,0.0001
3,Thursday-WorkingHours-Afternoon-Infilteration....,288602,18,0.0001
4,Monday-WorkingHours.pcap_ISCX.csv,529918,64,0.0002
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,28,0.0002
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,15,0.0001
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,4,0.0000


### 2.1.1 Identify Missing Values by Feature

The file-level analysis confirmed that missing values are present in the CICIDS-2017 source files, although their overall proportion is very small.

Before deciding on a cleaning strategy, the distribution of missing values across individual features is examined. This helps determine whether the missing values are concentrated in specific network-flow features and whether they affect a meaningful portion of the data.

The analysis is performed across all source files without modifying the original raw data.

In [ ]:
# Identify columns containing missing values in each source file

missing_by_column = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    column_missing = df.isna().sum()

    for column, count in column_missing.items():
        if count > 0:
            missing_by_column.append({
                "File Name": file,
                "Column": column,
                "Missing Values": count,
                "Missing Percentage": round(
                    (count / len(df)) * 100, 4
                )
            })

    del df

missing_by_column_df = pd.DataFrame(missing_by_column)

missing_by_column_df

,File Name,Column,Missing Values,Missing Percentage
0,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,1008,0.1455
1,Tuesday-WorkingHours.pcap_ISCX.csv,Flow Bytes/s,201,0.0451
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Flow Bytes/s,20,0.0117
3,Thursday-WorkingHours-Afternoon-Infilteration....,Flow Bytes/s,18,0.0062
4,Monday-WorkingHours.pcap_ISCX.csv,Flow Bytes/s,64,0.0121
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,Flow Bytes/s,28,0.0147
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Flow Bytes/s,15,0.0052
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Bytes/s,4,0.0018


### 2.1.2 Investigating the Cause of Missing Values

The missing-value analysis shows that all detected missing values occur in the `Flow Bytes/s` feature across the eight CICIDS-2017 source files.

In the initial dataset exploration, missing values in this feature were observed alongside zero values in `Flow Duration`. Since `Flow Bytes/s` represents the rate of bytes transferred per second, a zero flow duration makes this calculation undefined.

The same relationship is therefore investigated across all source files before selecting an appropriate treatment for these values.

In [ ]:
# Verify whether missing Flow Bytes/s values correspond to zero Flow Duration

flow_bytes_missing_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    missing_mask = df["Flow Bytes/s"].isna()

    missing_rows = df.loc[
        missing_mask,
        [" Flow Duration", "Total Length of Fwd Packets",
         " Total Length of Bwd Packets", "Flow Bytes/s"]
    ]

    flow_bytes_missing_analysis.append({
        "File Name": file,
        "Missing Flow Bytes/s": missing_mask.sum(),
        " Flow Duration = 0": (missing_rows[" Flow Duration"] == 0).sum(),
        "Other Flow Duration": (missing_rows[" Flow Duration"] != 0).sum()
    })

    del df

flow_bytes_missing_df = pd.DataFrame(flow_bytes_missing_analysis)

flow_bytes_missing_df

,File Name,Missing Flow Bytes/s,Flow Duration = 0,Other Flow Duration
0,Wednesday-workingHours.pcap_ISCX.csv,1008,1008,0
1,Tuesday-WorkingHours.pcap_ISCX.csv,201,201,0
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,20,20,0
3,Thursday-WorkingHours-Afternoon-Infilteration....,18,18,0
4,Monday-WorkingHours.pcap_ISCX.csv,64,64,0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,28,28,0
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,15,15,0
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,4,4,0


### Observation

The missing-value analysis reveals that all missing values occur exclusively in the `Flow Bytes/s` feature.

Further investigation shows that every record with a missing `Flow Bytes/s` value has a `Flow Duration` of zero. No missing `Flow Bytes/s` record was associated with a non-zero flow duration.

This pattern is consistent across all eight CICIDS-2017 source files and indicates that the missing values are caused by the undefined bytes-per-second calculation for zero-duration flows.

Before determining the final treatment, the corresponding packet-length values will be examined to determine whether these flows also contain zero transferred bytes.

In [ ]:
# Verify the total packet length for records with missing Flow Bytes/s

zero_duration_bytes_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    missing_mask = df["Flow Bytes/s"].isna()

    missing_rows = df.loc[
        missing_mask,
        [" Flow Duration",
         "Total Length of Fwd Packets",
         " Total Length of Bwd Packets"]
    ]

    total_bytes = (
        missing_rows["Total Length of Fwd Packets"] +
        missing_rows[" Total Length of Bwd Packets"]
    )

    zero_duration_bytes_analysis.append({
        "File Name": file,
        "Missing Flow Bytes/s": len(missing_rows),
        "Zero Total Bytes": (total_bytes == 0).sum(),
        "Non-Zero Total Bytes": (total_bytes != 0).sum(),
        "Maximum Total Bytes": total_bytes.max()
    })

    del df

zero_duration_bytes_df = pd.DataFrame(zero_duration_bytes_analysis)

zero_duration_bytes_df

,File Name,Missing Flow Bytes/s,Zero Total Bytes,Non-Zero Total Bytes,Maximum Total Bytes
0,Wednesday-workingHours.pcap_ISCX.csv,1008,1008,0,0
1,Tuesday-WorkingHours.pcap_ISCX.csv,201,201,0,0
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,20,20,0,0
3,Thursday-WorkingHours-Afternoon-Infilteration....,18,18,0,0
4,Monday-WorkingHours.pcap_ISCX.csv,64,64,0,0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,28,28,0,0
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,15,15,0,0
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,4,4,0,0


### Observation

The investigation confirms that all missing `Flow Bytes/s` values are associated with zero-duration flows.

For every source CSV file:

- All missing `Flow Bytes/s` records have `Flow Duration = 0`.
- All such records have zero total forward and backward packet bytes.
- No missing `Flow Bytes/s` record contains non-zero transferred bytes.

Therefore, these missing values represent zero-byte, zero-duration flows for which the bytes-per-second rate is mathematically undefined (`0 / 0`), rather than incomplete observations caused by data loss.

Since no actual network bytes were transferred in these records, the missing `Flow Bytes/s` values can be represented as `0` during data cleaning.

### Cleaning Decision

Missing values in `Flow Bytes/s` will be replaced with `0`.

This treatment is applied only after verifying the underlying flow duration and packet-byte values across all eight CICIDS-2017 source files.

## 2.2 Duplicate Record Analysis

Duplicate records can affect the statistical representation of network traffic and may introduce bias during model training.

Therefore, all eight CICIDS-2017 source files are examined independently for exact duplicate rows.

The analysis determines:

- The number of duplicate records in each source file.
- The percentage of duplicate records relative to the total number of rows.
- Whether duplicate records are present within individual source files.

Duplicates are analyzed before removal so that the cleaning decision is supported by evidence.

In [ ]:
# Analyze exact duplicate records across all CICIDS-2017 source files

duplicate_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    duplicate_count = df.duplicated().sum()
    total_rows = len(df)

    duplicate_analysis.append({
        "File Name": file,
        "Total Rows": total_rows,
        "Duplicate Rows": duplicate_count,
        "Duplicate Percentage": round(
            (duplicate_count / total_rows) * 100, 4
        )
    })

    del df

duplicate_analysis_df = pd.DataFrame(duplicate_analysis)

duplicate_analysis_df

,File Name,Total Rows,Duplicate Rows,Duplicate Percentage
0,Wednesday-workingHours.pcap_ISCX.csv,692703,81909,11.8245
1,Tuesday-WorkingHours.pcap_ISCX.csv,445909,24065,5.3968
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,6066,3.5606
3,Thursday-WorkingHours-Afternoon-Infilteration....,288602,35630,12.3457
4,Monday-WorkingHours.pcap_ISCX.csv,529918,26935,5.0829
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,6888,3.6057
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,72353,25.2570
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,2633,1.1664


### 2.2.1 Investigating Duplicate Records

The duplicate analysis shows that exact duplicate rows are present across all eight CICIDS-2017 source files, with substantial variation in their proportion.

These duplicates should not be assumed to be erroneous records. During the initial dataset exploration, repeated network flows were observed as a natural consequence of users repeatedly performing similar activities and generating recurring traffic patterns.

Therefore, the duplicate records are further investigated before any removal decision is made.

The investigation focuses on:

- The distribution of duplicate records across target labels.
- Whether duplicates are concentrated in specific traffic categories.
- Whether duplicate patterns are consistent with repeated network activity rather than accidental data duplication.

In [ ]:
# Analyze the target-label distribution of duplicate records

duplicate_label_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    # Identify exact duplicate rows
    duplicate_mask = df.duplicated(keep=False)
    duplicate_rows = df[duplicate_mask]

    # Count duplicates by target label
    label_counts = duplicate_rows[" Label"].value_counts()

    for label, count in label_counts.items():
        duplicate_label_analysis.append({
            "File Name": file,
            "Label": label,
            "Duplicate Rows": count
        })

    del df

duplicate_label_analysis_df = pd.DataFrame(duplicate_label_analysis)

duplicate_label_analysis_df

,File Name,Label,Duplicate Rows
0,Wednesday-workingHours.pcap_ISCX.csv,DoS Hulk,59564
1,Wednesday-workingHours.pcap_ISCX.csv,BENIGN,32236
2,Wednesday-workingHours.pcap_ISCX.csv,DoS slowloris,507
3,Wednesday-workingHours.pcap_ISCX.csv,DoS Slowhttptest,323
4,Wednesday-workingHours.pcap_ISCX.csv,DoS GoldenEye,14
5,Tuesday-WorkingHours.pcap_ISCX.csv,BENIGN,27115
6,Tuesday-WorkingHours.pcap_ISCX.csv,SSH-Patator,2826
7,Tuesday-WorkingHours.pcap_ISCX.csv,FTP-Patator,2457
8,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,BENIGN,8779
9,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Web Attack � Brute Force,62


### 2.2.2 Duplicate Frequency Analysis

The initial duplicate analysis identified substantial numbers of repeated rows across the CICIDS-2017 source files.

The target-label analysis shows that duplicate records occur in both benign and attack traffic. Therefore, duplicate records cannot be classified as erroneous solely based on their presence.

The frequency of repetition is further examined to determine whether duplicate records generally occur as isolated repeated flows or as highly repeated flow patterns.

This analysis will help determine the potential impact of duplicates on dataset representation and subsequent model training.

In [ ]:
# Analyze how frequently duplicate rows are repeated within each file

duplicate_frequency_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    # Count occurrences of each exact row
    row_counts = df.value_counts()

    # Keep only rows that occur more than once
    repeated_rows = row_counts[row_counts > 1]

    if len(repeated_rows) > 0:
        duplicate_frequency_analysis.append({
            "File Name": file,
            "Unique Duplicate Patterns": len(repeated_rows),
            "Maximum Repetitions": repeated_rows.max(),
            "Average Repetitions": round(repeated_rows.mean(), 2),
            "Repeated Rows": repeated_rows.sum()
        })
    else:
        duplicate_frequency_analysis.append({
            "File Name": file,
            "Unique Duplicate Patterns": 0,
            "Maximum Repetitions": 0,
            "Average Repetitions": 0,
            "Repeated Rows": 0
        })

    del df

duplicate_frequency_df = pd.DataFrame(duplicate_frequency_analysis)

duplicate_frequency_df


,File Name,Unique Duplicate Patterns,Maximum Repetitions,Average Repetitions,Repeated Rows
0,Wednesday-workingHours.pcap_ISCX.csv,10731,9329,8.54,91693
1,Tuesday-WorkingHours.pcap_ISCX.csv,8328,1551,3.88,32353
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,2774,359,3.19,8839
3,Thursday-WorkingHours-Afternoon-Infilteration....,9413,748,4.78,45040
4,Monday-WorkingHours.pcap_ISCX.csv,9933,3070,3.71,36860
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,3224,555,3.14,10112
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,35351,461,3.05,107704
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,1307,118,3.01,3940


### 2.2.3 Cross-Label Consistency of Duplicate Records

Duplicate records were found across both benign and attack traffic. Before deciding how duplicate records should be treated, it is necessary to determine whether identical network-flow records are consistently associated with the same target label.

If an identical feature record occurs with multiple labels, removing duplicates blindly could discard potentially important class information and may indicate label inconsistency within the source data.

Therefore, duplicate feature records are examined for cross-label conflicts.

In [ ]:
# Check whether identical feature records occur with different labels

cross_label_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    # Exclude the target label from the feature comparison
    feature_columns = [col for col in df.columns if col != " Label"]

    # Group identical feature records and count distinct labels
    label_counts = (
        df.groupby(feature_columns, dropna=False)[" Label"]
        .nunique()
    )

    # Number of feature patterns associated with multiple labels
    conflicting_patterns = (label_counts > 1).sum()

    cross_label_analysis.append({
        "File Name": file,
        "Unique Feature Patterns": len(label_counts),
        "Cross-Label Conflicts": conflicting_patterns
    })

    del df

cross_label_analysis_df = pd.DataFrame(cross_label_analysis)

cross_label_analysis_df

,File Name,Unique Feature Patterns,Cross-Label Conflicts
0,Wednesday-workingHours.pcap_ISCX.csv,610746,48
1,Tuesday-WorkingHours.pcap_ISCX.csv,421844,0
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,164300,0
3,Thursday-WorkingHours-Afternoon-Infilteration....,252972,0
4,Monday-WorkingHours.pcap_ISCX.csv,502983,0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,184145,0
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,214114,0
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,223112,0


### Key Observation — Cross-Label Duplicate Consistency

The cross-label consistency analysis identified **zero feature patterns associated with multiple target labels across all eight CICIDS-2017 source files**.

This indicates that the repeated records observed in the previous analysis are label-consistent within each source file. Therefore, the presence of repeated rows does not appear to introduce direct target-label conflicts in the analyzed data.

However, the high frequency of repeated network-flow records indicates that these repetitions should not be treated as simple data-entry errors. They may represent repeated observations of similar or identical network-flow behavior generated during traffic capture.

Therefore, duplicate records will not be removed blindly at this stage. Their treatment will instead be considered in the context of model training and evaluation, particularly because indiscriminate duplicate removal could alter the natural distribution of network traffic patterns.

**Decision:** Retain the duplicate records during the initial cleaning stage and document their presence for consideration during dataset splitting and model evaluation.

## 2.3 Data-Type Consistency

Before applying cleaning and feature engineering operations, the data types of the CICIDS-2017 features are examined across all eight source files.

Consistent data types are important because the dataset contains numerical network-flow measurements as well as identifier and categorical fields. Unexpected type differences between source files could lead to incorrect numerical operations or inconsistent preprocessing.

This analysis focuses on:

- Comparing column data types across all source files
- Identifying columns with inconsistent data types
- Identifying numerical features that were not interpreted as numeric
- Verifying the data type of the target `Label` column

No type conversion is performed at this stage. The purpose is to identify potential data-quality issues before deciding on the appropriate cleaning operation.

### 2.3.1 Data-Type Comparison Across Source Files

Each CICIDS-2017 source file is inspected independently and the inferred data type of every column is recorded.

Since all eight files follow the same feature schema established during dataset validation, differences in inferred data types would indicate potential formatting or representation inconsistencies.

In [ ]:
# Analyze data types across all CICIDS-2017 source files

dtype_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    for column in df.columns:
        dtype_analysis.append({
            "File Name": file,
            "Column": column,
            "Data Type": str(df[column].dtype)
        })

    del df

dtype_analysis_df = pd.DataFrame(dtype_analysis)

dtype_analysis_df.head()

,File Name,Column,Data Type
0,Wednesday-workingHours.pcap_ISCX.csv,Destination Port,int64
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Duration,int64
2,Wednesday-workingHours.pcap_ISCX.csv,Total Fwd Packets,int64
3,Wednesday-workingHours.pcap_ISCX.csv,Total Backward Packets,int64
4,Wednesday-workingHours.pcap_ISCX.csv,Total Length of Fwd Packets,int64


### 2.3.2 Identifying Inconsistent Column Data Types

The data types of corresponding columns are compared across the eight source files.

A column is considered potentially inconsistent if the same feature is represented using different inferred data types in different source files.

In [ ]:
# Identify columns with inconsistent data types across source files

dtype_comparison = (
    dtype_analysis_df
    .groupby("Column")["Data Type"]
    .nunique()
    .reset_index(name="Number of Different Data Types")
)

inconsistent_columns = dtype_comparison[
    dtype_comparison["Number of Different Data Types"] > 1
]

inconsistent_columns

,Column,Number of Different Data Types


### 2.3.3 Numerical Feature Validation

Most CICIDS-2017 network-flow features represent quantitative measurements such as packet counts, durations, lengths, rates, and statistical characteristics.

These features are expected to be represented using numerical data types. The analysis therefore identifies non-numerical columns so that identifier and categorical fields can be distinguished from numerical features before feature engineering.

In [ ]:
# Inspect the data types using the first source file as the schema reference

reference_file = csv_files[0]
reference_path = os.path.join(DATA_PATH, reference_file)

reference_df = pd.read_csv(reference_path)

dtype_summary = (
    reference_df.dtypes
    .value_counts()
    .rename_axis("Data Type")
    .reset_index(name="Number of Columns")
)

dtype_summary

,Data Type,Number of Columns
0,int64,54
1,float64,24
2,object,1


In [ ]:
# List columns that are not numerical

non_numeric_columns = reference_df.select_dtypes(
    exclude=["number"]
).columns.tolist()

non_numeric_columns

[' Label']

### 2.3.4 Target Label Data-Type and Value Consistency

The `Label` column represents the target variable for intrusion detection. Its data type and categorical values are therefore checked across all source files.

The analysis verifies whether the target is consistently represented and whether unexpected formatting differences such as leading or trailing whitespace are present.

In [ ]:
# Inspect Label data type and formatting across all source files

label_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    label_series = df[" Label"].astype(str)

    label_analysis.append({
        "File Name": file,
        "Label Data Type": str(df[" Label"].dtype),
        "Unique Labels": df[" Label"].nunique(),
        "Labels With Leading/Trailing Spaces": (
            label_series != label_series.str.strip()
        ).sum()
    })

    del df

label_analysis_df = pd.DataFrame(label_analysis)

label_analysis_df

,File Name,Label Data Type,Unique Labels,Labels With Leading/Trailing Spaces
0,Wednesday-workingHours.pcap_ISCX.csv,object,6,0
1,Tuesday-WorkingHours.pcap_ISCX.csv,object,3,0
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,object,4,0
3,Thursday-WorkingHours-Afternoon-Infilteration....,object,2,0
4,Monday-WorkingHours.pcap_ISCX.csv,object,1,0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,object,2,0
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,object,2,0
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,object,2,0


### 2.3.5 Key Observations

The data-type analysis shows that the feature representation is consistent across all eight CICIDS-2017 source files.

The dataset contains 54 integer features and 24 floating-point features, while the `Label` column is the only non-numerical feature. The `Label` column is consistently represented as an object type across all source files.

No columns were found with conflicting inferred data types between source files. Furthermore, no leading or trailing whitespace was detected in the target labels.

The differences in the number of unique labels between source files are expected because the source files represent different traffic and attack scenarios.

Therefore, no data-type correction is required at this stage. The existing numerical and categorical representations can be retained for subsequent preprocessing and feature engineering.

## 2.4 Invalid and Anomalous Value Analysis

After verifying data-type consistency, the numerical features are examined for invalid or anomalous values.

The analysis focuses on identifying:

- Infinite values
- Negative values in inherently non-negative network-flow features
- Extremely large values that may indicate anomalous observations
- Other numerical irregularities that could affect preprocessing or model training

The purpose of this analysis is to distinguish genuine network behaviour from values that are mathematically or semantically invalid.

### 2.4.1 Infinite Value Analysis

Numerical network-flow features should not contain positive or negative infinity because such values can interfere with statistical analysis, scaling, and machine-learning algorithms.

Each source file is therefore checked for positive and negative infinite values.

In [ ]:
# Check for infinite values across all source files

infinity_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    numeric_df = df.select_dtypes(include=["number"])

    positive_infinity = np.isposinf(numeric_df).sum().sum()
    negative_infinity = np.isneginf(numeric_df).sum().sum()

    infinity_analysis.append({
        "File Name": file,
        "Positive Infinity": positive_infinity,
        "Negative Infinity": negative_infinity,
        "Total Infinity": positive_infinity + negative_infinity
    })

    del df

infinity_analysis_df = pd.DataFrame(infinity_analysis)

infinity_analysis_df

,File Name,Positive Infinity,Negative Infinity,Total Infinity
0,Wednesday-workingHours.pcap_ISCX.csv,1586,0,1586
1,Tuesday-WorkingHours.pcap_ISCX.csv,327,0,327
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,250,0,250
3,Thursday-WorkingHours-Afternoon-Infilteration....,396,0,396
4,Monday-WorkingHours.pcap_ISCX.csv,810,0,810
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,216,0,216
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,727,0,727
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,64,0,64


### 2.4.2 Investigating the Source of Infinite Values

The infinite-value analysis identified positive infinity values across all eight CICIDS-2017 source files, with no negative infinity values.

Before applying any cleaning operation, the source of these infinite values must be determined. In particular, features representing rates may become infinite when their denominator is zero.

The columns containing infinite values are therefore identified first, followed by an examination of the corresponding flow-duration and byte-related features.

In [ ]:
# Identify the columns containing infinite values in each source file

infinity_column_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    numeric_df = df.select_dtypes(include=["number"])

    for column in numeric_df.columns:
        positive_inf = np.isposinf(numeric_df[column]).sum()
        negative_inf = np.isneginf(numeric_df[column]).sum()

        if positive_inf > 0 or negative_inf > 0:
            infinity_column_analysis.append({
                "File Name": file,
                "Column": column,
                "Positive Infinity": positive_inf,
                "Negative Infinity": negative_inf,
                "Total Infinity": positive_inf + negative_inf
            })

    del df

infinity_column_analysis_df = pd.DataFrame(infinity_column_analysis)

infinity_column_analysis_df

,File Name,Column,Positive Infinity,Negative Infinity,Total Infinity
0,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,289,0,289
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Packets/s,1297,0,1297
2,Tuesday-WorkingHours.pcap_ISCX.csv,Flow Bytes/s,63,0,63
3,Tuesday-WorkingHours.pcap_ISCX.csv,Flow Packets/s,264,0,264
4,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Flow Bytes/s,115,0,115
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Flow Packets/s,135,0,135
6,Thursday-WorkingHours-Afternoon-Infilteration....,Flow Bytes/s,189,0,189
7,Thursday-WorkingHours-Afternoon-Infilteration....,Flow Packets/s,207,0,207
8,Monday-WorkingHours.pcap_ISCX.csv,Flow Bytes/s,373,0,373
9,Monday-WorkingHours.pcap_ISCX.csv,Flow Packets/s,437,0,437


In [ ]:
# Verify whether infinite rate values correspond to zero Flow Duration

infinity_rate_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    # Rows containing infinity in either rate feature
    infinity_mask = (
        np.isinf(df["Flow Bytes/s"]) |
        np.isinf(df[" Flow Packets/s"])
    )

    infinity_rows = df.loc[
        infinity_mask,
        [
            " Flow Duration",
            " Total Fwd Packets",
            " Total Backward Packets",
            "Total Length of Fwd Packets",
            " Total Length of Bwd Packets",
            "Flow Bytes/s",
            " Flow Packets/s"
        ]
    ]

    infinity_rate_analysis.append({
        "File Name": file,
        "Infinite Rate Rows": len(infinity_rows),
        "Zero Flow Duration": (
            infinity_rows[" Flow Duration"] == 0
        ).sum(),
        "Non-Zero Flow Duration": (
            infinity_rows[" Flow Duration"] != 0
        ).sum()
    })

    del df

infinity_rate_analysis_df = pd.DataFrame(infinity_rate_analysis)

infinity_rate_analysis_df

,File Name,Infinite Rate Rows,Zero Flow Duration,Non-Zero Flow Duration
0,Wednesday-workingHours.pcap_ISCX.csv,1297,1297,0
1,Tuesday-WorkingHours.pcap_ISCX.csv,264,264,0
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,135,135,0
3,Thursday-WorkingHours-Afternoon-Infilteration....,207,207,0
4,Monday-WorkingHours.pcap_ISCX.csv,437,437,0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,122,122,0
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,371,371,0
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,34,34,0


The analysis confirms that all rows containing infinite rate values have a `Flow Duration` of zero across all eight source files.

The corresponding packet-count and packet-length features are therefore examined to determine whether these rows contain actual network traffic activity. This allows the infinite values to be distinguished between mathematically undefined rate calculations and potentially invalid observations.

In [ ]:
# Examine packet and byte activity in rows with infinite rate values

infinity_activity_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)

    infinity_mask = (
        np.isinf(df["Flow Bytes/s"]) |
        np.isinf(df[" Flow Packets/s"])
    )

    infinity_rows = df.loc[
        infinity_mask,
        [
            " Flow Duration",
            " Total Fwd Packets",
            " Total Backward Packets",
            "Total Length of Fwd Packets",
            " Total Length of Bwd Packets"
        ]
    ]

    total_packets = (
        infinity_rows[" Total Fwd Packets"] +
        infinity_rows[" Total Backward Packets"]
    )

    total_bytes = (
        infinity_rows["Total Length of Fwd Packets"] +
        infinity_rows[" Total Length of Bwd Packets"]
    )

    infinity_activity_analysis.append({
        "File Name": file,
        "Infinite Rate Rows": len(infinity_rows),
        "Zero Total Packets": (total_packets == 0).sum(),
        "Non-Zero Total Packets": (total_packets > 0).sum(),
        "Zero Total Bytes": (total_bytes == 0).sum(),
        "Non-Zero Total Bytes": (total_bytes > 0).sum()
    })

    del df

infinity_activity_analysis_df = pd.DataFrame(
    infinity_activity_analysis
)

infinity_activity_analysis_df

,File Name,Infinite Rate Rows,Zero Total Packets,Non-Zero Total Packets,Zero Total Bytes,Non-Zero Total Bytes
0,Wednesday-workingHours.pcap_ISCX.csv,1297,0,1297,1008,289
1,Tuesday-WorkingHours.pcap_ISCX.csv,264,0,264,201,63
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,135,0,135,20,115
3,Thursday-WorkingHours-Afternoon-Infilteration....,207,0,207,18,189
4,Monday-WorkingHours.pcap_ISCX.csv,437,0,437,64,373
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,122,0,122,28,94
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,371,0,371,15,356
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,34,0,34,4,30


In [ ]:
# Analyze the two infinite rate features separately

infinity_rate_detail = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    # Flow Bytes/s
    bytes_inf = np.isinf(df["Flow Bytes/s"])

    bytes_rows = df.loc[
        bytes_inf,
        [
            "Flow Duration",
            "Total Fwd Packets",
            "Total Backward Packets",
            "Total Length of Fwd Packets",
            "Total Length of Bwd Packets",
            "Flow Bytes/s"
        ]
    ]

    # Flow Packets/s
    packets_inf = np.isinf(df["Flow Packets/s"])

    packets_rows = df.loc[
        packets_inf,
        [
            "Flow Duration",
            "Total Fwd Packets",
            "Total Backward Packets",
            "Flow Packets/s"
        ]
    ]

    infinity_rate_detail.append({
        "File Name": file,

        "Infinite Flow Bytes/s": len(bytes_rows),
        "Flow Bytes/s with Zero Duration":
            (bytes_rows["Flow Duration"] == 0).sum(),
        "Flow Bytes/s with Non-Zero Bytes": (
            (
                bytes_rows["Total Length of Fwd Packets"] +
                bytes_rows["Total Length of Bwd Packets"]
            ) > 0
        ).sum(),

        "Infinite Flow Packets/s": len(packets_rows),
        "Flow Packets/s with Zero Duration":
            (packets_rows["Flow Duration"] == 0).sum(),
        "Flow Packets/s with Non-Zero Packets": (
            (
                packets_rows["Total Fwd Packets"] +
                packets_rows["Total Backward Packets"]
            ) > 0
        ).sum()
    })

    del df

infinity_rate_detail_df = pd.DataFrame(infinity_rate_detail)

infinity_rate_detail_df

,File Name,Infinite Flow Bytes/s,Flow Bytes/s with Zero Duration,Flow Bytes/s with Non-Zero Bytes,Infinite Flow Packets/s,Flow Packets/s with Zero Duration,Flow Packets/s with Non-Zero Packets
0,Wednesday-workingHours.pcap_ISCX.csv,289,289,289,1297,1297,1297
1,Tuesday-WorkingHours.pcap_ISCX.csv,63,63,63,264,264,264
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,115,115,115,135,135,135
3,Thursday-WorkingHours-Afternoon-Infilteration....,189,189,189,207,207,207
4,Monday-WorkingHours.pcap_ISCX.csv,373,373,373,437,437,437
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,94,94,94,122,122,122
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,356,356,356,371,371,371
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,30,30,30,34,34,34


### Finding: Infinite Rate Values

The analysis shows that all infinite values in `Flow Bytes/s` and `Flow Packets/s` occur when `Flow Duration` is zero.

For `Flow Bytes/s`, every infinite-rate row has a positive total number of bytes transferred. Similarly, every infinite `Flow Packets/s` value occurs in a row with a positive total packet count.

Therefore, the infinite values are a consequence of division by zero rather than arbitrary data corruption. These rows represent zero-duration flows with recorded network activity.

The infinite values will therefore be treated as invalid numerical values during the cleaning stage. The underlying flow records will not be removed solely because their rate features are infinite; instead, the affected rate values will be handled while preserving the remaining flow information.

### 2.4.3 Negative Value Analysis

Negative values are examined in numerical network-flow features to identify observations that may be mathematically or semantically invalid.

Several CICIDS-2017 features represent quantities that are inherently non-negative, such as flow duration, packet counts, byte counts, and traffic rates. Negative values in such features may therefore indicate data-quality issues or anomalous observations.

The analysis is performed in multiple stages to determine:

- which numerical features contain negative values,
- the frequency and magnitude of the negative values,
- the context in which they occur, and
- whether they are associated with particular traffic classes.

The purpose is to understand the nature and distribution of negative values before deciding how they should be handled during data cleaning.

In [ ]:
# Check for negative values in numerical features

negative_value_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numeric_columns = df.select_dtypes(include=["number"]).columns

    negative_counts = (df[numeric_columns] < 0).sum()

    affected_columns = negative_counts[negative_counts > 0]

    negative_value_analysis.append({
        "File Name": file,
        "Negative Values": int(affected_columns.sum()),
        "Affected Columns": len(affected_columns),
        "Columns": affected_columns.index.tolist()
    })

    del df

negative_value_analysis_df = pd.DataFrame(negative_value_analysis)

negative_value_analysis_df

,File Name,Negative Values,Affected Columns,Columns
0,Wednesday-workingHours.pcap_ISCX.csv,539591,12,"[Flow Duration, Flow Bytes/s, Flow Packets/s, ..."
1,Tuesday-WorkingHours.pcap_ISCX.csv,457338,11,"[Flow Duration, Flow Packets/s, Flow IAT Mean,..."
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,184601,8,"[Flow Duration, Flow Bytes/s, Flow Packets/s, ..."
3,Thursday-WorkingHours-Afternoon-Infilteration....,267157,8,"[Flow Duration, Flow Bytes/s, Flow Packets/s, ..."
4,Monday-WorkingHours.pcap_ISCX.csv,524256,12,"[Flow Duration, Flow Bytes/s, Flow Packets/s, ..."
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,214522,8,"[Flow Duration, Flow Bytes/s, Flow Packets/s, ..."
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,137508,8,"[Flow Duration, Flow Bytes/s, Flow Packets/s, ..."
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,121348,9,"[Flow Duration, Flow Bytes/s, Flow Packets/s, ..."


In [ ]:
# Examine the negative values in each affected numerical feature

negative_value_details = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numeric_columns = df.select_dtypes(include=["number"]).columns

    for column in numeric_columns:
        negative_mask = df[column] < 0

        if negative_mask.any():
            negative_values = df.loc[negative_mask, column]

            negative_value_details.append({
                "File Name": file,
                "Column": column,
                "Negative Count": len(negative_values),
                "Minimum Negative Value": negative_values.min(),
                "Maximum Negative Value": negative_values.max(),
                "Unique Negative Values": negative_values.nunique()
            })

    del df

negative_value_details_df = pd.DataFrame(negative_value_details)

negative_value_details_df

,File Name,Column,Negative Count,Minimum Negative Value,Maximum Negative Value,Unique Negative Values
0,Wednesday-workingHours.pcap_ISCX.csv,Flow Duration,21,-1.0,-1.0,1
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,16,-12000000.0,-12000000.0,1
2,Wednesday-workingHours.pcap_ISCX.csv,Flow Packets/s,21,-2000000.0,-2000000.0,1
3,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Mean,21,-1.0,-1.0,1
4,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Max,21,-1.0,-1.0,1
...,...,...,...,...,...,...
71,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Max,2,-1.0,-1.0,1
72,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Min,108,-12.0,-1.0,4
73,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Fwd IAT Min,6,-12.0,-1.0,2
74,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Init_Win_bytes_forward,32925,-1.0,-1.0,1


In [ ]:
# Investigate the frequency of specific negative values

negative_value_frequency = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numeric_columns = df.select_dtypes(include=["number"]).columns

    for column in numeric_columns:

        negative_values = df.loc[df[column] < 0, column]

        if len(negative_values) > 0:

            value_counts = negative_values.value_counts()

            for value, count in value_counts.items():
                negative_value_frequency.append({
                    "File Name": file,
                    "Column": column,
                    "Negative Value": value,
                    "Frequency": count
                })

    del df

negative_value_frequency_df = pd.DataFrame(
    negative_value_frequency
)

negative_value_frequency_df

,File Name,Column,Negative Value,Frequency
0,Wednesday-workingHours.pcap_ISCX.csv,Flow Duration,-1.0,21
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,-12000000.0,16
2,Wednesday-workingHours.pcap_ISCX.csv,Flow Packets/s,-2000000.0,21
3,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Mean,-1.0,21
4,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Max,-1.0,21
...,...,...,...,...
221,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Min,-3.0,1
222,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Fwd IAT Min,-1.0,5
223,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Fwd IAT Min,-12.0,1
224,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Init_Win_bytes_forward,-1.0,32925


In [ ]:
# Investigate the context of negative values

negative_context_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numeric_columns = df.select_dtypes(include=["number"]).columns

    for column in numeric_columns:

        negative_mask = df[column] < 0

        if negative_mask.any():

            negative_rows = df.loc[negative_mask]

            total_packets = (
                negative_rows["Total Fwd Packets"] +
                negative_rows["Total Backward Packets"]
            )

            total_bytes = (
                negative_rows["Total Length of Fwd Packets"] +
                negative_rows["Total Length of Bwd Packets"]
            )

            negative_context_analysis.append({
                "File Name": file,
                "Column": column,
                "Negative Rows": len(negative_rows),

                "Zero Flow Duration": (
                    negative_rows["Flow Duration"] == 0
                ).sum(),

                "Negative Flow Duration": (
                    negative_rows["Flow Duration"] < 0
                ).sum(),

                "Positive Flow Duration": (
                    negative_rows["Flow Duration"] > 0
                ).sum(),

                "Zero Total Packets": (
                    total_packets == 0
                ).sum(),

                "Non-Zero Total Packets": (
                    total_packets > 0
                ).sum(),

                "Zero Total Bytes": (
                    total_bytes == 0
                ).sum(),

                "Non-Zero Total Bytes": (
                    total_bytes > 0
                ).sum()
            })

    del df

negative_context_analysis_df = pd.DataFrame(
    negative_context_analysis
)

negative_context_analysis_df

,File Name,Column,Negative Rows,Zero Flow Duration,Negative Flow Duration,Positive Flow Duration,Zero Total Packets,Non-Zero Total Packets,Zero Total Bytes,Non-Zero Total Bytes
0,Wednesday-workingHours.pcap_ISCX.csv,Flow Duration,21,0,21,0,0,21,5,16
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,16,0,16,0,0,16,0,16
2,Wednesday-workingHours.pcap_ISCX.csv,Flow Packets/s,21,0,21,0,0,21,5,16
3,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Mean,21,0,21,0,0,21,5,16
4,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Max,21,0,21,0,0,21,5,16
...,...,...,...,...,...,...,...,...,...,...
71,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Max,2,0,2,0,0,2,0,2
72,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Min,108,0,2,106,0,108,0,108
73,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Fwd IAT Min,6,0,0,6,0,6,0,6
74,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Init_Win_bytes_forward,32925,0,0,32925,0,32925,54,32871


In [ ]:
# Examine labels associated with negative values
# to determine whether negative values are concentrated in particular traffic classes

negative_label_analysis = []

features_to_check = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Min",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward"
]

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    for column in features_to_check:

        if column not in df.columns:
            continue

        negative_mask = df[column] < 0

        if negative_mask.any():

            label_counts = (
                df.loc[negative_mask, "Label"]
                .value_counts()
            )

            for label, count in label_counts.items():

                negative_label_analysis.append({
                    "File Name": file,
                    "Column": column,
                    "Label": label,
                    "Negative Rows": count
                })

    del df

negative_label_analysis_df = pd.DataFrame(
    negative_label_analysis
)

negative_label_analysis_df

,File Name,Column,Label,Negative Rows
0,Wednesday-workingHours.pcap_ISCX.csv,Flow Duration,BENIGN,21
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,BENIGN,16
2,Wednesday-workingHours.pcap_ISCX.csv,Flow Packets/s,BENIGN,21
3,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Mean,BENIGN,21
4,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Max,BENIGN,21
...,...,...,...,...
79,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Min,DDoS,19
80,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Fwd IAT Min,DDoS,6
81,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Init_Win_bytes_forward,BENIGN,32925
82,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Init_Win_bytes_backward,DDoS,46528


### 2.4.4 Investigating the Context of Negative Values

The negative-value analysis identified negative observations across multiple numerical features. The context of these observations is therefore examined to determine whether the negative values occur alongside meaningful network activity or whether they are associated with other anomalous feature values.

For each affected feature, the corresponding flow duration, packet counts, and byte counts are examined. This helps determine whether the negative values represent isolated inconsistencies or occur systematically with particular flow characteristics.

The purpose of this analysis is to provide sufficient evidence for deciding how the affected values should be treated during the subsequent data-cleaning stage.

In [ ]:
# Examine the context of negative values using related flow activity features

negative_context_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    # Identify numerical columns containing negative values
    numerical_columns = df.select_dtypes(include=np.number).columns

    for column in numerical_columns:

        negative_mask = df[column] < 0

        if negative_mask.sum() == 0:
            continue

        negative_rows = df.loc[negative_mask]

        negative_context_analysis.append({
            "File Name": file,
            "Column": column,
            "Negative Rows": len(negative_rows),

            "Zero Flow Duration": (
                negative_rows["Flow Duration"] == 0
            ).sum(),

            "Non-Zero Flow Duration": (
                negative_rows["Flow Duration"] != 0
            ).sum(),

            "Zero Total Packets": (
                (
                    negative_rows["Total Fwd Packets"] +
                    negative_rows["Total Backward Packets"]
                ) == 0
            ).sum(),

            "Non-Zero Total Packets": (
                (
                    negative_rows["Total Fwd Packets"] +
                    negative_rows["Total Backward Packets"]
                ) > 0
            ).sum(),

            "Zero Total Bytes": (
                (
                    negative_rows["Total Length of Fwd Packets"] +
                    negative_rows["Total Length of Bwd Packets"]
                ) == 0
            ).sum(),

            "Non-Zero Total Bytes": (
                (
                    negative_rows["Total Length of Fwd Packets"] +
                    negative_rows["Total Length of Bwd Packets"]
                ) > 0
            ).sum()
        })

    del df

negative_context_analysis_df = pd.DataFrame(
    negative_context_analysis
)

negative_context_analysis_df

,File Name,Column,Negative Rows,Zero Flow Duration,Non-Zero Flow Duration,Zero Total Packets,Non-Zero Total Packets,Zero Total Bytes,Non-Zero Total Bytes
0,Wednesday-workingHours.pcap_ISCX.csv,Flow Duration,21,0,21,0,21,5,16
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,16,0,16,0,16,0,16
2,Wednesday-workingHours.pcap_ISCX.csv,Flow Packets/s,21,0,21,0,21,5,16
3,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Mean,21,0,21,0,21,5,16
4,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Max,21,0,21,0,21,5,16
...,...,...,...,...,...,...,...,...,...
71,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Max,2,0,2,0,2,0,2
72,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Min,108,0,108,0,108,0,108
73,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Fwd IAT Min,6,0,6,0,6,0,6
74,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Init_Win_bytes_forward,32925,0,32925,0,32925,54,32871


In [ ]:
# Classify negative-valued observations based on independent flow activity features

negative_activity_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numerical_columns = df.select_dtypes(include=np.number).columns

    for column in numerical_columns:

        negative_mask = df[column] < 0

        if negative_mask.sum() == 0:
            continue

        negative_rows = df.loc[negative_mask].copy()

        total_packets = (
            negative_rows["Total Fwd Packets"] +
            negative_rows["Total Backward Packets"]
        )

        total_bytes = (
            negative_rows["Total Length of Fwd Packets"] +
            negative_rows["Total Length of Bwd Packets"]
        )

        # Flow activity is determined independently of the affected feature
        active_flow_mask = (
            (total_packets > 0) &
            (total_bytes > 0)
        )

        zero_activity_mask = (
            (total_packets == 0) |
            (total_bytes == 0)
        )

        negative_activity_analysis.append({
            "File Name": file,
            "Column": column,
            "Negative Rows": len(negative_rows),
            "Active Flow Rows": active_flow_mask.sum(),
            "Zero-Activity Rows": zero_activity_mask.sum()
        })

    del df

negative_activity_analysis_df = pd.DataFrame(
    negative_activity_analysis
)

negative_activity_analysis_df

,File Name,Column,Negative Rows,Active Flow Rows,Zero-Activity Rows
0,Wednesday-workingHours.pcap_ISCX.csv,Flow Duration,21,16,5
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,16,16,0
2,Wednesday-workingHours.pcap_ISCX.csv,Flow Packets/s,21,16,5
3,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Mean,21,16,5
4,Wednesday-workingHours.pcap_ISCX.csv,Flow IAT Max,21,16,5
...,...,...,...,...,...
71,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Max,2,2,0
72,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Min,108,108,0
73,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Fwd IAT Min,6,6,0
74,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Init_Win_bytes_forward,32925,32871,54


In [ ]:
# Summarize whether negative values occur in active or zero-activity flows

negative_activity_summary = (
    negative_activity_analysis_df
    .groupby("Column")
    .agg(
        Total_Negative_Rows=("Negative Rows", "sum"),
        Active_Flow_Rows=("Active Flow Rows", "sum"),
        Zero_Activity_Rows=("Zero-Activity Rows", "sum")
    )
    .reset_index()
)

negative_activity_summary["Active Flow Percentage"] = (
    negative_activity_summary["Active_Flow_Rows"] /
    negative_activity_summary["Total_Negative_Rows"] * 100
)

negative_activity_summary["Zero-Activity Percentage"] = (
    negative_activity_summary["Zero_Activity_Rows"] /
    negative_activity_summary["Total_Negative_Rows"] * 100
)

negative_activity_summary.sort_values(
    "Total_Negative_Rows",
    ascending=False
)

,Column,Total_Negative_Rows,Active_Flow_Rows,Zero_Activity_Rows,Active Flow Percentage,Zero-Activity Percentage
10,Init_Win_bytes_backward,1441552,1281901,159651,88.925061,11.074939
11,Init_Win_bytes_forward,1001189,999483,1706,99.829603,0.170397
5,Flow IAT Min,2891,2805,86,97.025251,2.974749
4,Flow IAT Mean,115,85,30,73.913043,26.086957
2,Flow Duration,115,85,30,73.913043,26.086957
3,Flow IAT Max,115,85,30,73.913043,26.086957
6,Flow Packets/s,115,85,30,73.913043,26.086957
1,Flow Bytes/s,85,85,0,100.000000,0.000000
8,Fwd Header Length.1,35,35,0,100.000000,0.000000
12,min_seg_size_forward,35,35,0,100.000000,0.000000


### 2.4.4.1 Negative Values Across Traffic Classes

Negative values are further examined with respect to traffic labels to determine whether they are concentrated within specific traffic classes. This helps distinguish dataset-specific patterns from values that occur broadly across different types of network traffic.

In [ ]:
# Summarize negative values by feature and traffic label

negative_label_summary = (
    negative_label_analysis_df
    .groupby(["Column", "Label"])["Negative Rows"]
    .sum()
    .reset_index()
    .sort_values(
        ["Column", "Negative Rows"],
        ascending=[True, False]
    )
)

negative_label_summary

,Column,Label,Negative Rows
0,Flow Bytes/s,BENIGN,85
1,Flow Duration,BENIGN,115
2,Flow IAT Max,BENIGN,115
3,Flow IAT Mean,BENIGN,115
4,Flow IAT Min,BENIGN,2697
7,Flow IAT Min,DoS Hulk,159
5,Flow IAT Min,DDoS,19
6,Flow IAT Min,DoS GoldenEye,5
8,Flow IAT Min,FTP-Patator,4
9,Flow IAT Min,Heartbleed,4


### 2.4.4.2 Interpretation of Negative Values Across Traffic Classes

The label-wise analysis shows that negative values occur in both BENIGN and attack traffic classes. However, their distribution is highly feature-dependent. The Init_Win_bytes_forward and Init_Win_bytes_backward features contain the largest number of negative observations, with the majority occurring in BENIGN traffic. Negative values in Flow IAT Min are distributed across both BENIGN and several attack classes. Other affected features, including Flow Duration, Flow Bytes/s, Flow Packets/s, Flow IAT Mean, and Flow IAT Max, show negative values primarily within the BENIGN class.

Therefore, the presence of negative values cannot be attributed solely to malicious traffic. Since several affected features represent quantities that are expected to be non-negative, these observations require further investigation before deciding whether they should be retained, transformed, or treated as invalid values.

In [ ]:
# Summarize the number of traffic classes containing negative values
# for each affected feature

negative_class_summary = (
    negative_label_summary
    .groupby("Column")
    .agg(
        Traffic_Classes=("Label", "nunique"),
        Total_Negative_Rows=("Negative Rows", "sum")
    )
    .reset_index()
    .sort_values(
        "Total_Negative_Rows",
        ascending=False
    )
)

negative_class_summary

,Column,Traffic_Classes,Total_Negative_Rows
7,Init_Win_bytes_backward,11,1441552
8,Init_Win_bytes_forward,2,1001189
4,Flow IAT Min,8,2891
3,Flow IAT Mean,1,115
1,Flow Duration,1,115
5,Flow Packets/s,1,115
2,Flow IAT Max,1,115
0,Flow Bytes/s,1,85
6,Fwd IAT Min,3,17


### 2.4.4.3 Interpretation of Negative Value Analysis

The analysis shows that negative values are distributed differently across numerical features and traffic classes. The largest concentration occurs in Init_Win_bytes_forward and Init_Win_bytes_backward, which contain approximately 1.0 million and 1.44 million negative observations respectively. These values occur across multiple traffic classes, with BENIGN traffic accounting for the majority of observations. Flow IAT Min also contains negative values across several traffic classes, whereas the remaining affected features contain substantially fewer negative observations.

Since negative values occur in both BENIGN and attack traffic, they cannot be interpreted simply as indicators of malicious activity. Furthermore, several affected features represent quantities that are normally expected to be non-negative. Therefore, these observations should be treated as potential invalid or anomalous values during subsequent preprocessing rather than being used directly for model training.

### 2.4.5 Extremely Large Value Analysis

After examining infinite and negative values, the numerical features are further examined for extremely large finite values.

Extremely large values are not automatically considered invalid because network-flow data can naturally contain large packet counts, byte counts, or timing-related values. However, unusually large observations may indicate anomalous records, measurement issues, or values that could disproportionately influence statistical analysis and machine-learning models.

The analysis therefore identifies the upper range of numerical features and investigates features containing unusually large observations before deciding whether any preprocessing treatment is necessary.

In [ ]:
# Examine the maximum value of each numerical feature
# across all source files

large_value_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numeric_df = df.select_dtypes(include=[np.number])

    max_values = numeric_df.max()

    for column, value in max_values.items():
        large_value_analysis.append({
            "File Name": file,
            "Column": column,
            "Maximum Value": value
        })

    del df

large_value_analysis_df = pd.DataFrame(
    large_value_analysis
)

large_value_analysis_df

,File Name,Column,Maximum Value
0,Wednesday-workingHours.pcap_ISCX.csv,Destination Port,65487.0
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Duration,119999998.0
2,Wednesday-workingHours.pcap_ISCX.csv,Total Fwd Packets,203943.0
3,Wednesday-workingHours.pcap_ISCX.csv,Total Backward Packets,272353.0
4,Wednesday-workingHours.pcap_ISCX.csv,Total Length of Fwd Packets,1224076.0
...,...,...,...
619,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Active Min,100000000.0
620,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Idle Mean,120000000.0
621,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Idle Std,65300000.0
622,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Idle Max,120000000.0


### 2.4.5.1 Upper Percentile Analysis

Maximum values alone are insufficient to determine whether an observation is anomalously large because a feature may naturally contain values spanning a wide numerical range.

Therefore, the upper percentiles of the numerical features are examined to determine how the largest observations compare with the general distribution of each feature.

The 95th, 99th, and 99.9th percentiles are used to identify features in which a small number of observations may be substantially larger than the majority of the data.

In [ ]:
# Examine upper percentiles of finite numerical values
# to identify potentially extreme observations

percentile_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numeric_df = df.select_dtypes(include=[np.number])

    # Exclude infinite values from percentile calculations
    numeric_df = numeric_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    for column in numeric_df.columns:

        percentile_analysis.append({
            "File Name": file,
            "Column": column,
            "95th Percentile": numeric_df[column].quantile(0.95),
            "99th Percentile": numeric_df[column].quantile(0.99),
            "99.9th Percentile": numeric_df[column].quantile(0.999),
            "Maximum Finite Value": numeric_df[column].max()
        })

    del df

percentile_analysis_df = pd.DataFrame(
    percentile_analysis
)

percentile_analysis_df

,File Name,Column,95th Percentile,99th Percentile,99.9th Percentile,Maximum Finite Value
0,Wednesday-workingHours.pcap_ISCX.csv,Destination Port,52296.0,5.876600e+04,6.137030e+04,65487.0
1,Wednesday-workingHours.pcap_ISCX.csv,Flow Duration,102636993.5,1.177202e+08,1.197526e+08,119999998.0
2,Wednesday-workingHours.pcap_ISCX.csv,Total Fwd Packets,16.0,4.200000e+01,2.420000e+02,203943.0
3,Wednesday-workingHours.pcap_ISCX.csv,Total Backward Packets,15.0,4.600000e+01,3.580000e+02,272353.0
4,Wednesday-workingHours.pcap_ISCX.csv,Total Length of Fwd Packets,1630.0,1.000100e+04,2.909919e+04,1224076.0
...,...,...,...,...,...,...
619,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Active Min,1377968.8,4.125899e+06,4.938219e+06,100000000.0
620,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Idle Mean,71500000.0,9.395600e+07,1.000000e+08,120000000.0
621,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Idle Std,43600000.0,5.710000e+07,6.270000e+07,65300000.0
622,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Idle Max,81500000.0,9.470000e+07,1.000000e+08,120000000.0


### 2.4.5.2 Extreme Value Ratio Analysis

To further investigate potentially anomalous large values, the maximum finite value of each numerical feature is compared with its 99.9th percentile.

A large ratio between the maximum and the 99.9th percentile indicates that the maximum may represent an isolated extreme observation rather than a value commonly observed within the feature distribution.

This analysis is used to identify candidate features for further investigation. The identified values are not immediately removed because extreme values may represent legitimate network behaviour.

In [ ]:
# Compare maximum finite values with the 99.9th percentile
# to identify potentially isolated extreme observations

extreme_value_analysis = percentile_analysis_df.copy()

# Calculate the ratio only where the 99.9th percentile is positive
extreme_value_analysis["Maximum / 99.9th Ratio"] = np.where(
    extreme_value_analysis["99.9th Percentile"] > 0,
    extreme_value_analysis["Maximum Finite Value"] /
    extreme_value_analysis["99.9th Percentile"],
    np.nan
)

# Remove cases where the ratio cannot be interpreted
extreme_value_analysis = extreme_value_analysis.dropna(
    subset=["Maximum / 99.9th Ratio"]
)

extreme_value_analysis = extreme_value_analysis.sort_values(
    "Maximum / 99.9th Ratio",
    ascending=False
)

extreme_value_analysis

,File Name,Column,95th Percentile,99th Percentile,99.9th Percentile,Maximum Finite Value,Maximum / 99.9th Ratio
68,Wednesday-workingHours.pcap_ISCX.csv,act_data_pkt_fwd,9.0,25.00,165.000,197124.0,1194.690909
65,Wednesday-workingHours.pcap_ISCX.csv,Subflow Bwd Bytes,11595.0,53276.66,599830.302,627046409.0,1045.373011
5,Wednesday-workingHours.pcap_ISCX.csv,Total Length of Bwd Packets,11595.0,53276.66,599830.302,627000000.0,1045.295641
380,Monday-WorkingHours.pcap_ISCX.csv,act_data_pkt_fwd,12.0,33.00,211.083,213557.0,1011.720508
317,Monday-WorkingHours.pcap_ISCX.csv,Total Length of Bwd Packets,7387.0,79315.66,747833.974,655453030.0,876.468645
...,...,...,...,...,...,...,...
67,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_backward,2076.0,40958.00,65535.000,65535.0,1.000000
125,Tuesday-WorkingHours.pcap_ISCX.csv,ACK Flag Count,1.0,1.00,1.000,1.0,1.000000
126,Tuesday-WorkingHours.pcap_ISCX.csv,URG Flag Count,1.0,1.00,1.000,1.0,1.000000
66,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_forward,29200.0,65311.96,65535.000,65535.0,1.000000


### 2.4.5.3 Investigation of Extreme Observations

The extreme-value ratio analysis identified several numerical features whose maximum values are substantially larger than their upper percentiles. These observations require further investigation before any cleaning operation is performed.

The investigation focuses on:

- Identifying the source rows containing the extreme values
- Examining the corresponding traffic labels
- Comparing related network-flow features
- Determining whether the extreme values represent valid network behaviour or potentially erroneous observations

The objective is to distinguish genuine high-magnitude network-flow observations from values that may require correction or removal during preprocessing.

In [ ]:
# Identify the most extreme feature observations for further investigation

extreme_candidates = extreme_value_analysis[
    extreme_value_analysis["Maximum / 99.9th Ratio"] >= 100
].copy()

extreme_candidates = extreme_candidates.sort_values(
    "Maximum / 99.9th Ratio",
    ascending=False
)

extreme_candidates

,File Name,Column,95th Percentile,99th Percentile,99.9th Percentile,Maximum Finite Value,Maximum / 99.9th Ratio
68,Wednesday-workingHours.pcap_ISCX.csv,act_data_pkt_fwd,9.0,25.00,1.650000e+02,197124.0,1194.690909
65,Wednesday-workingHours.pcap_ISCX.csv,Subflow Bwd Bytes,11595.0,53276.66,5.998303e+05,627046409.0,1045.373011
5,Wednesday-workingHours.pcap_ISCX.csv,Total Length of Bwd Packets,11595.0,53276.66,5.998303e+05,627000000.0,1045.295641
380,Monday-WorkingHours.pcap_ISCX.csv,act_data_pkt_fwd,12.0,33.00,2.110830e+02,213557.0,1011.720508
317,Monday-WorkingHours.pcap_ISCX.csv,Total Length of Bwd Packets,7387.0,79315.66,7.478340e+05,655453030.0,876.468645
377,Monday-WorkingHours.pcap_ISCX.csv,Subflow Bwd Bytes,7387.0,79315.66,7.478340e+05,655453030.0,876.468645
62,Wednesday-workingHours.pcap_ISCX.csv,Subflow Fwd Packets,16.0,42.00,2.420000e+02,203943.0,842.739669
2,Wednesday-workingHours.pcap_ISCX.csv,Total Fwd Packets,16.0,42.00,2.420000e+02,203943.0,842.739669
314,Monday-WorkingHours.pcap_ISCX.csv,Total Fwd Packets,20.0,52.00,2.760000e+02,219759.0,796.228261
374,Monday-WorkingHours.pcap_ISCX.csv,Subflow Fwd Packets,20.0,52.00,2.760000e+02,219759.0,796.228261


In [ ]:
# Identify the source rows containing the most extreme observations

extreme_row_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    # Select extreme candidates belonging to the current file
    file_candidates = extreme_candidates[
        extreme_candidates["File Name"] == file
    ]

    if file_candidates.empty:
        continue

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    for _, candidate in file_candidates.iterrows():

        column = candidate["Column"]
        maximum_value = candidate["Maximum Finite Value"]

        if column not in df.columns:
            continue

        matching_rows = df[
            df[column] == maximum_value
        ]

        for row_index, row in matching_rows.iterrows():

            extreme_row_analysis.append({
                "File Name": file,
                "Row Index": row_index,
                "Column": column,
                "Extreme Value": maximum_value,
                "Label": row["Label"]
            })

    del df

extreme_row_analysis_df = pd.DataFrame(
    extreme_row_analysis
)

extreme_row_analysis_df

,File Name,Row Index,Column,Extreme Value,Label
0,Wednesday-workingHours.pcap_ISCX.csv,721,act_data_pkt_fwd,197124.0,BENIGN
1,Wednesday-workingHours.pcap_ISCX.csv,12453,Subflow Bwd Bytes,627046409.0,BENIGN
2,Wednesday-workingHours.pcap_ISCX.csv,12453,Total Length of Bwd Packets,627000000.0,BENIGN
3,Wednesday-workingHours.pcap_ISCX.csv,721,Subflow Fwd Packets,203943.0,BENIGN
4,Wednesday-workingHours.pcap_ISCX.csv,721,Total Fwd Packets,203943.0,BENIGN
5,Wednesday-workingHours.pcap_ISCX.csv,6526,Total Backward Packets,272353.0,BENIGN
6,Wednesday-workingHours.pcap_ISCX.csv,6526,Subflow Bwd Packets,272353.0,BENIGN
7,Wednesday-workingHours.pcap_ISCX.csv,6526,Fwd Header Length.1,4290372.0,BENIGN
8,Wednesday-workingHours.pcap_ISCX.csv,6526,Fwd Header Length,4290372.0,BENIGN
9,Wednesday-workingHours.pcap_ISCX.csv,6526,Bwd Header Length,5447060.0,BENIGN


In [ ]:
# Examine related network-flow features for the identified extreme observations
# to determine whether the extreme values are internally consistent

related_feature_analysis = []

related_features = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Subflow Fwd Packets",
    "Subflow Bwd Packets",
    "Subflow Fwd Bytes",
    "Subflow Bwd Bytes",
    "Fwd Header Length",
    "Fwd Header Length.1",
    "Bwd Header Length",
    "act_data_pkt_fwd",
    "Label"
]

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    # Select extreme observations belonging to the current file
    file_extreme_rows = extreme_row_analysis_df[
        extreme_row_analysis_df["File Name"] == file
    ]

    if file_extreme_rows.empty:
        continue

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    available_features = [
        column for column in related_features
        if column in df.columns
    ]

    for row_index in file_extreme_rows["Row Index"].unique():

        if row_index >= len(df):
            continue

        row = df.iloc[int(row_index)]

        analysis_row = {
            "File Name": file,
            "Row Index": row_index
        }

        for column in available_features:
            analysis_row[column] = row[column]

        related_feature_analysis.append(analysis_row)

    del df

related_feature_analysis_df = pd.DataFrame(
    related_feature_analysis
)

related_feature_analysis_df

,File Name,Row Index,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Subflow Fwd Packets,Subflow Bwd Packets,Subflow Fwd Bytes,Subflow Bwd Bytes,Fwd Header Length,Fwd Header Length.1,Bwd Header Length,act_data_pkt_fwd,Label
0,Wednesday-workingHours.pcap_ISCX.csv,721,119999933,203943,263412,1224076,584000000,203943,263412,1224076,583510015,4202748,4202748,5268240,197124,BENIGN
1,Wednesday-workingHours.pcap_ISCX.csv,12453,56306996,161754,183913,955929,627000000,161754,183913,955929,627046409,3269516,3269516,3678284,159059,BENIGN
2,Wednesday-workingHours.pcap_ISCX.csv,6526,119959890,202456,272353,1196931,593000000,202456,272353,1196931,593469129,4290372,4290372,5447060,192326,BENIGN
3,Tuesday-WorkingHours.pcap_ISCX.csv,43222,65138900,177702,228838,1315,627000000,177702,228838,1315,627039032,3609208,3609208,4576772,4,BENIGN
4,Tuesday-WorkingHours.pcap_ISCX.csv,14740,119977366,206446,276072,42720,615000000,206446,276072,42720,615111383,4225596,4225596,5521440,113,BENIGN
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,9544,119996804,200755,270686,1197199,591000000,200755,270686,1197199,590596141,4173072,4173072,5413720,192491,BENIGN
6,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,40153,59415092,155812,179415,917301,627000000,155812,179415,917301,627040569,3162564,3162564,3588312,152621,BENIGN
7,Thursday-WorkingHours-Afternoon-Infilteration....,103957,116123990,9453,4828,12900000,32710,9453,4828,12870338,32710,302496,302496,154496,9130,BENIGN
8,Thursday-WorkingHours-Afternoon-Infilteration....,93457,31600032,22673,44553,1502,63600000,22673,44553,1502,63586401,742984,742984,1425704,6,BENIGN
9,Monday-WorkingHours.pcap_ISCX.csv,4768,119999876,219759,291922,1323378,655453030,219759,291922,1323378,655453030,4480016,4480016,5838440,213557,BENIGN


In [ ]:
# Check whether the identified extreme observations are duplicated
# within their respective source files

extreme_duplicate_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    file_extreme_rows = extreme_row_analysis_df[
        extreme_row_analysis_df["File Name"] == file
    ]

    if file_extreme_rows.empty:
        continue

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    extreme_indices = (
        file_extreme_rows["Row Index"]
        .astype(int)
        .unique()
    )

    extreme_rows = df.loc[
        df.index.intersection(extreme_indices)
    ]

    duplicate_flags = extreme_rows.duplicated(
        keep=False
    )

    extreme_duplicate_analysis.append({
        "File Name": file,
        "Extreme Rows Examined": len(extreme_rows),
        "Duplicated Extreme Rows": duplicate_flags.sum(),
        "Unique Extreme Rows": (
            (~duplicate_flags).sum()
        )
    })

    del df

extreme_duplicate_analysis_df = pd.DataFrame(
    extreme_duplicate_analysis
)

extreme_duplicate_analysis_df

,File Name,Extreme Rows Examined,Duplicated Extreme Rows,Unique Extreme Rows
0,Wednesday-workingHours.pcap_ISCX.csv,3,0,3
1,Tuesday-WorkingHours.pcap_ISCX.csv,2,0,2
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,2,0,2
3,Thursday-WorkingHours-Afternoon-Infilteration....,2,0,2
4,Monday-WorkingHours.pcap_ISCX.csv,2,0,2
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,2,0,2


In [ ]:
# Check for other invalid numerical values within the identified extreme observations
# before determining whether the extreme observations should be retained

extreme_invalid_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    file_extreme_rows = extreme_row_analysis_df[
        extreme_row_analysis_df["File Name"] == file
    ]

    if file_extreme_rows.empty:
        continue

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    extreme_indices = (
        file_extreme_rows["Row Index"]
        .astype(int)
        .unique()
    )

    extreme_rows = df.loc[
        df.index.intersection(extreme_indices)
    ]

    numeric_extreme_rows = extreme_rows.select_dtypes(
        include=[np.number]
    )

    extreme_invalid_analysis.append({
        "File Name": file,
        "Extreme Rows": len(extreme_rows),
        "NaN Values": numeric_extreme_rows.isna().sum().sum(),
        "Positive Infinity": np.isposinf(
            numeric_extreme_rows.to_numpy()
        ).sum(),
        "Negative Infinity": np.isneginf(
            numeric_extreme_rows.to_numpy()
        ).sum(),
        "Negative Values": (
            numeric_extreme_rows < 0
        ).sum().sum()
    })

    del df

extreme_invalid_analysis_df = pd.DataFrame(
    extreme_invalid_analysis
)

extreme_invalid_analysis_df

,File Name,Extreme Rows,NaN Values,Positive Infinity,Negative Infinity,Negative Values
0,Wednesday-workingHours.pcap_ISCX.csv,3,0,0,0,3
1,Tuesday-WorkingHours.pcap_ISCX.csv,2,0,0,0,2
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,2,0,0,0,2
3,Thursday-WorkingHours-Afternoon-Infilteration....,2,0,0,0,1
4,Monday-WorkingHours.pcap_ISCX.csv,2,0,0,0,2
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,2,0,0,0,2


In [ ]:
# Identify the negative-valued features within the extreme observations
# to determine whether they correspond to previously investigated anomalies

extreme_negative_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    file_extreme_rows = extreme_row_analysis_df[
        extreme_row_analysis_df["File Name"] == file
    ]

    if file_extreme_rows.empty:
        continue

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    extreme_indices = (
        file_extreme_rows["Row Index"]
        .astype(int)
        .unique()
    )

    extreme_rows = df.loc[
        df.index.intersection(extreme_indices)
    ]

    numeric_extreme_rows = extreme_rows.select_dtypes(
        include=[np.number]
    )

    for row_index, row in numeric_extreme_rows.iterrows():

        negative_columns = row[row < 0]

        for column, value in negative_columns.items():

            extreme_negative_analysis.append({
                "File Name": file,
                "Row Index": row_index,
                "Column": column,
                "Negative Value": value,
                "Label": df.loc[row_index, "Label"]
            })

    del df

extreme_negative_analysis_df = pd.DataFrame(
    extreme_negative_analysis
)

extreme_negative_analysis_df

,File Name,Row Index,Column,Negative Value,Label
0,Wednesday-workingHours.pcap_ISCX.csv,721,Flow IAT Min,-11.0,BENIGN
1,Wednesday-workingHours.pcap_ISCX.csv,6526,Flow IAT Min,-1.0,BENIGN
2,Wednesday-workingHours.pcap_ISCX.csv,12453,Flow IAT Min,-2.0,BENIGN
3,Tuesday-WorkingHours.pcap_ISCX.csv,14740,Flow IAT Min,-1.0,BENIGN
4,Tuesday-WorkingHours.pcap_ISCX.csv,43222,Flow IAT Min,-1.0,BENIGN
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,9544,Flow IAT Min,-2.0,BENIGN
6,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,40153,Flow IAT Min,-5.0,BENIGN
7,Thursday-WorkingHours-Afternoon-Infilteration....,93457,Flow IAT Min,-1.0,BENIGN
8,Monday-WorkingHours.pcap_ISCX.csv,4768,Flow IAT Min,-11.0,BENIGN
9,Monday-WorkingHours.pcap_ISCX.csv,17373,Flow IAT Min,-5.0,BENIGN


### 2.4.5.4 Conclusion of Extreme Value Investigation

The investigation of extreme numerical observations identified several unusually large values across packet counts, byte counts, header lengths, and related network-flow features.

Further examination showed that these extreme values:

Occur across multiple related network-flow features within the same observations
Maintain consistency between corresponding packet and byte-related features
Are associated with BENIGN traffic in the identified extreme observations
Are not duplicated records
Contain no missing values or infinite values
Do not exhibit negative values in the extreme-magnitude features themselves


The only negative values observed within these extreme rows occur in the Flow IAT Min feature, which is consistent with the negative-value patterns identified earlier in the negative-value analysis.

Therefore, the identified extreme observations are considered valid high-magnitude network-flow observations rather than automatically invalid or corrupted records. They should not be removed solely based on their magnitude. The associated negative Flow IAT Min values will be addressed as part of the negative-value handling process.

### 2.4.6 Cleaning Decisions for Invalid and Anomalous Values

The preceding analyses identified three major categories of invalid or anomalous numerical values: infinite values, negative values, and extremely large finite values.

Before applying any cleaning operation, the findings from these analyses are consolidated into explicit treatment decisions.

| Issue | Finding | Cleaning Decision |
|---|---|---|
| Infinite values | Infinite values occur in rate features when `Flow Duration` is zero, while the corresponding flow records contain network activity. | Treat the infinite feature values as invalid numerical values while preserving the underlying flow records. |
| Negative values | Negative values occur across several numerical features and traffic classes. Their occurrence is feature-dependent and should not be interpreted as an indication of malicious traffic. | Do not remove entire rows solely because of negative values. Feature-specific treatment will be applied during the cleaning stage. |
| Extremely large values | Extreme observations are consistent across related packet, byte, and header features, are not duplicated, and do not contain other invalid numerical values. | Retain the extreme observations because their magnitude alone does not provide sufficient evidence that they are erroneous. |

These decisions ensure that data cleaning is based on the observed characteristics and context of the dataset rather than on arbitrary numerical thresholds.

The actual cleaning operations will be applied in the subsequent data-cleaning stage, followed by post-cleaning validation.

## 3. Data Cleaning

This stage applies the cleaning decisions established during the data quality assessment.

The cleaning process will be performed consistently across all available CICIDS-2017 CSV files while preserving valid network-flow observations.

The main cleaning operations are:

1. Replace invalid infinite numerical values with `NaN`.
2. Handle feature-specific invalid negative values identified during assessment.
3. Preserve valid extreme observations unless there is sufficient evidence that they are erroneous.
4. Remove duplicate records where appropriate.
5. Standardize column names and data types.
6. Validate the cleaned files before consolidation.

The objective is to produce a consistent and reliable dataset for subsequent feature engineering and model development.

### 3.1 Applying Basic Numerical Cleaning

Infinite values identified during the data quality assessment are converted to missing values (`NaN`). This prevents invalid numerical values from propagating into subsequent feature engineering and model training.

The original flow records are retained during this operation.

In [ ]:
# Replace infinite numerical values with NaN
# while preserving the original dataset structure

infinity_cleaning_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numeric_df = df.select_dtypes(include=[np.number])

    positive_infinity_count = np.isposinf(numeric_df).sum().sum()
    negative_infinity_count = np.isneginf(numeric_df).sum().sum()

    total_infinity_count = (
        positive_infinity_count
        + negative_infinity_count
    )

    infinity_cleaning_analysis.append({
        "File Name": file,
        "Positive Infinity": positive_infinity_count,
        "Negative Infinity": negative_infinity_count,
        "Total Infinity": total_infinity_count
    })

    del df

infinity_cleaning_analysis_df = pd.DataFrame(
    infinity_cleaning_analysis
)

infinity_cleaning_analysis_df

,File Name,Positive Infinity,Negative Infinity,Total Infinity
0,Wednesday-workingHours.pcap_ISCX.csv,1586,0,1586
1,Tuesday-WorkingHours.pcap_ISCX.csv,327,0,327
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,250,0,250
3,Thursday-WorkingHours-Afternoon-Infilteration....,396,0,396
4,Monday-WorkingHours.pcap_ISCX.csv,810,0,810
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,216,0,216
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,727,0,727
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,64,0,64


In [ ]:
# Replace positive and negative infinity values with NaN
# to allow them to be handled during subsequent missing-value treatment

infinity_replacement_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    positive_infinity_count = np.isposinf(df.select_dtypes(include=[np.number])).sum().sum()
    negative_infinity_count = np.isneginf(df.select_dtypes(include=[np.number])).sum().sum()

    df = df.replace([np.inf, -np.inf], np.nan)

    remaining_infinity_count = np.isinf(
        df.select_dtypes(include=[np.number])
    ).sum().sum()

    infinity_replacement_analysis.append({
        "File Name": file,
        "Positive Infinity Replaced": positive_infinity_count,
        "Negative Infinity Replaced": negative_infinity_count,
        "Remaining Infinity": remaining_infinity_count
    })

    del df

infinity_replacement_analysis_df = pd.DataFrame(
    infinity_replacement_analysis
)

infinity_replacement_analysis_df

,File Name,Positive Infinity Replaced,Negative Infinity Replaced,Remaining Infinity
0,Wednesday-workingHours.pcap_ISCX.csv,1586,0,0
1,Tuesday-WorkingHours.pcap_ISCX.csv,327,0,0
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,250,0,0
3,Thursday-WorkingHours-Afternoon-Infilteration....,396,0,0
4,Monday-WorkingHours.pcap_ISCX.csv,810,0,0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,216,0,0
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,727,0,0
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,64,0,0


### 2.4.7 Treatment of Negative Values

Negative values identified during the data quality assessment were reviewed according to their feature semantics and traffic context. Negative inter-arrival-time values are not physically meaningful because time intervals cannot be negative. Therefore, the affected feature values are treated as invalid and replaced with NaN, while the corresponding network-flow records are retained for subsequent missing-value handling.

In [ ]:
# Replace invalid negative numerical values with NaN
# while retaining the corresponding network-flow records

negative_value_treatment = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numeric_columns = df.select_dtypes(include=[np.number]).columns

    negative_mask = df[numeric_columns] < 0

    negative_values_before = negative_mask.sum().sum()

    df[numeric_columns] = df[numeric_columns].mask(
        negative_mask,
        np.nan
    )

    negative_values_remaining = (
        (df[numeric_columns] < 0)
        .sum()
        .sum()
    )

    negative_value_treatment.append({
        "File Name": file,
        "Negative Values Replaced": negative_values_before,
        "Negative Values Remaining": negative_values_remaining
    })

    del df

negative_value_treatment_df = pd.DataFrame(
    negative_value_treatment
)

negative_value_treatment_df

,File Name,Negative Values Replaced,Negative Values Remaining
0,Wednesday-workingHours.pcap_ISCX.csv,539591,0
1,Tuesday-WorkingHours.pcap_ISCX.csv,457338,0
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,184601,0
3,Thursday-WorkingHours-Afternoon-Infilteration....,267157,0
4,Monday-WorkingHours.pcap_ISCX.csv,524256,0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,214522,0
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,137508,0
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,121348,0


### 2.4.7.1 Column-wise Negative Value Assessment

The initial negative-value treatment revealed a large number of negative numerical values across the dataset. Since negative values may have different meanings depending on the feature, a column-wise assessment is performed before applying any replacement. This ensures that only semantically invalid negative values are treated as anomalous rather than modifying legitimate feature values.

In [ ]:
# Analyze negative values column-wise
# before applying any final treatment

negative_column_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    numeric_columns = df.select_dtypes(include=[np.number]).columns

    for column in numeric_columns:

        negative_count = (df[column] < 0).sum()

        if negative_count > 0:
            negative_column_analysis.append({
                "File Name": file,
                "Column": column,
                "Negative Values": negative_count,
                "Minimum Value": df[column].min()
            })

    del df

negative_column_analysis_df = pd.DataFrame(
    negative_column_analysis
)

negative_column_analysis_df = negative_column_analysis_df.sort_values(
    "Negative Values",
    ascending=False
)

negative_column_analysis_df

,File Name,Column,Negative Values,Minimum Value
10,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_backward,335475,-1.0
49,Monday-WorkingHours.pcap_ISCX.csv,Init_Win_bytes_backward,299094,-1.0
21,Tuesday-WorkingHours.pcap_ISCX.csv,Init_Win_bytes_backward,256599,-1.0
48,Monday-WorkingHours.pcap_ISCX.csv,Init_Win_bytes_forward,224495,-1.0
9,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_forward,203253,-1.0
...,...,...,...,...
67,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Duration,2,-1.0
68,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Bytes/s,2,-12000000.0
11,Wednesday-workingHours.pcap_ISCX.csv,min_seg_size_forward,1,-1.0
7,Wednesday-workingHours.pcap_ISCX.csv,Fwd Header Length,1,-11.0


### 2.4.7.2 Negative Value Treatment

The column-wise assessment shows that negative values occur in features where negative measurements are not physically meaningful. The Init_Win_bytes_forward and Init_Win_bytes_backward features contain a large number of -1 values, which are treated as unavailable/sentinel values rather than valid measurements. Other negative observations occur in flow duration, inter-arrival time, packet rate, byte rate, header length, and segment-size features, where negative values are invalid. Therefore, negative values in these identified numerical features are converted to NaN for subsequent missing-value handling and imputation.

In [18]:
# Replace identified invalid negative values with NaN
# for features where negative values are not valid measurements

negative_value_treatment = []

negative_columns = [
    "Init_Win_bytes_backward",
    "Init_Win_bytes_forward",
    "Flow IAT Min",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Fwd IAT Min",
    "Fwd Header Length",
    "Fwd Header Length.1",
    "Bwd Header Length",
    "min_seg_size_forward"
]

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    replaced_count = 0

    for column in negative_columns:

        if column in df.columns:

            negative_mask = df[column] < 0

            replaced_count += negative_mask.sum()

            df.loc[negative_mask, column] = np.nan

    remaining_negative_values = (
        df[negative_columns]
        .select_dtypes(include=[np.number])
        .lt(0)
        .sum()
        .sum()
    )

    negative_value_treatment.append({
        "File Name": file,
        "Negative Values Replaced": replaced_count,
        "Negative Values Remaining": remaining_negative_values
    })

    del df

negative_value_treatment_df = pd.DataFrame(
    negative_value_treatment
)

negative_value_treatment_df

,File Name,Negative Values Replaced,Negative Values Remaining
0,Wednesday-workingHours.pcap_ISCX.csv,539591,0
1,Tuesday-WorkingHours.pcap_ISCX.csv,457338,0
2,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,184601,0
3,Thursday-WorkingHours-Afternoon-Infilteration....,267157,0
4,Monday-WorkingHours.pcap_ISCX.csv,524256,0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,214522,0
6,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,137508,0
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,121348,0


### 2.4.7.3 Negative Value Treatment Summary

The treatment successfully removed all identified negative values from the selected numerical features across the dataset. The large number of replacements is primarily attributable to -1 sentinel values in the Init_Win_bytes_forward and Init_Win_bytes_backward features. These values are treated as unavailable observations and converted to NaN, while negative values in flow timing, rate, header-length, and segment-size features are treated as invalid observations. The resulting values will be handled during subsequent missing-value processing.

In [19]:
# Summarize negative values identified by feature
# across all dataset files

negative_feature_summary = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    for column in negative_columns:

        if column in df.columns:

            negative_count = (df[column] < 0).sum()

            if negative_count > 0:
                negative_feature_summary.append({
                    "File Name": file,
                    "Column": column,
                    "Negative Values Identified": negative_count,
                    "Minimum Value": df[column].min()
                })

    del df

negative_feature_summary_df = pd.DataFrame(
    negative_feature_summary
)

negative_feature_summary_df.sort_values(
    "Negative Values Identified",
    ascending=False
)

,File Name,Column,Negative Values Identified,Minimum Value
0,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_backward,335475,-1.0
39,Monday-WorkingHours.pcap_ISCX.csv,Init_Win_bytes_backward,299094,-1.0
12,Tuesday-WorkingHours.pcap_ISCX.csv,Init_Win_bytes_backward,256599,-1.0
40,Monday-WorkingHours.pcap_ISCX.csv,Init_Win_bytes_forward,224495,-1.0
1,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_forward,203253,-1.0
...,...,...,...,...
70,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Mean,2,-1.0
74,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Packets/s,2,-2000000.0
11,Wednesday-workingHours.pcap_ISCX.csv,min_seg_size_forward,1,-1.0
9,Wednesday-workingHours.pcap_ISCX.csv,Fwd Header Length,1,-11.0


### 2.4.7.4 Negative Value Analysis — Conclusion

The column-wise analysis identified negative observations in several numerical network-flow features. A large proportion of these observations occurred as -1 in Init_Win_bytes_forward and Init_Win_bytes_backward, while other negative values were observed in flow duration, inter-arrival time, packet/byte rate, header length, and segment-size features. Since negative values are not valid measurements for these selected features, they were treated as invalid observations and converted to NaN. After treatment, no negative values remained in the selected features across the dataset files. These resulting NaN values will be addressed during subsequent missing-value handling.

### 2.4.8 Post-Treatment Missing Value Assessment

Invalid infinite and negative observations identified during the data-quality analysis have been converted to NaN values for subsequent handling. A post-treatment assessment is therefore performed to quantify the resulting missing values across all dataset files and numerical features. This assessment provides the basis for the subsequent missing-value treatment stage.

In [20]:
# Assess missing values after invalid-value treatment
# to determine the remaining data-quality requirements

post_treatment_missing_analysis = []

for file in csv_files:
    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    # Convert infinite values to NaN
    df = df.replace([np.inf, -np.inf], np.nan)

    # Convert identified negative values to NaN
    for column in negative_columns:

        if column in df.columns:
            df.loc[df[column] < 0, column] = np.nan

    for column in df.columns:

        missing_count = df[column].isna().sum()

        if missing_count > 0:
            post_treatment_missing_analysis.append({
                "File Name": file,
                "Column": column,
                "Missing Values": missing_count,
                "Missing Percentage": (
                    missing_count / len(df) * 100
                )
            })

    del df

post_treatment_missing_analysis_df = pd.DataFrame(
    post_treatment_missing_analysis
)

post_treatment_missing_analysis_df = (
    post_treatment_missing_analysis_df
    .sort_values(
        "Missing Values",
        ascending=False
    )
    .reset_index(drop=True)
)

post_treatment_missing_analysis_df

,File Name,Column,Missing Values,Missing Percentage
0,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_backward,335475,48.429847
1,Monday-WorkingHours.pcap_ISCX.csv,Init_Win_bytes_backward,299094,56.441563
2,Tuesday-WorkingHours.pcap_ISCX.csv,Init_Win_bytes_backward,256599,57.545149
3,Monday-WorkingHours.pcap_ISCX.csv,Init_Win_bytes_forward,224495,42.364102
4,Wednesday-workingHours.pcap_ISCX.csv,Init_Win_bytes_forward,203253,29.342012
...,...,...,...,...
72,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Duration,2,0.000886
73,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow IAT Mean,2,0.000886
74,Wednesday-workingHours.pcap_ISCX.csv,min_seg_size_forward,1,0.000144
75,Wednesday-workingHours.pcap_ISCX.csv,Fwd Header Length,1,0.000144


### 2.4.9 Dataset-Level Missing Value Assessment

Missing values are aggregated across all CICIDS-2017 dataset files to evaluate the overall availability of each feature. This provides a consolidated view of feature completeness and supports evidence-based decisions regarding feature retention, removal, or subsequent missing-value treatment.

In [21]:
# Dataset-level missing value assessment
# Aggregates missing values across all CICIDS-2017 files

dataset_missing_summary = {}

for file in csv_files:

    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    # Apply the same invalid-value treatment used earlier
    df = df.replace([np.inf, -np.inf], np.nan)

    for column in negative_columns:
        if column in df.columns:
            df.loc[df[column] < 0, column] = np.nan

    for column in df.columns:

        if column not in dataset_missing_summary:
            dataset_missing_summary[column] = {
                "Total Rows": 0,
                "Missing Values": 0
            }

        dataset_missing_summary[column]["Total Rows"] += len(df)
        dataset_missing_summary[column]["Missing Values"] += (
            df[column].isna().sum()
        )

    del df


dataset_missing_analysis = []

for column, values in dataset_missing_summary.items():

    total_rows = values["Total Rows"]
    missing_values = values["Missing Values"]

    if missing_values > 0:

        missing_percentage = (
            missing_values / total_rows
        ) * 100

        dataset_missing_analysis.append({
            "Column": column,
            "Total Rows": total_rows,
            "Missing Values": missing_values,
            "Missing Percentage": missing_percentage
        })


dataset_missing_analysis_df = pd.DataFrame(
    dataset_missing_analysis
)

dataset_missing_analysis_df = (
    dataset_missing_analysis_df
    .sort_values(
        "Missing Percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

dataset_missing_analysis_df

,Column,Total Rows,Missing Values,Missing Percentage
0,Init_Win_bytes_backward,2830743,1441552,50.924863
1,Init_Win_bytes_forward,2830743,1001189,35.368417
2,Flow Packets/s,2830743,2982,0.105343
3,Flow Bytes/s,2830743,2952,0.104284
4,Flow IAT Min,2830743,2891,0.102129
5,Flow IAT Max,2830743,115,0.004063
6,Flow IAT Mean,2830743,115,0.004063
7,Flow Duration,2830743,115,0.004063
8,Fwd Header Length,2830743,35,0.001236
9,Fwd Header Length.1,2830743,35,0.001236


### 2.4.10 Final Data Quality Decision

The dataset-level assessment shows that most missing values introduced during invalid-value treatment occur at a very low proportion of the overall dataset. These features can therefore be retained and handled through appropriate missing-value imputation during the cleaning stage.

However, `Init_Win_bytes_forward` and `Init_Win_bytes_backward` exhibit substantially higher missing-value proportions of approximately 35.37% and 50.92%, respectively. Retaining these features would require extensive imputation and may introduce artificial patterns into the dataset. Therefore, these two features are excluded from the modelling dataset.

The remaining features are retained because their missing-value proportions are negligible. Their missing values will be handled using an appropriate numerical imputation strategy.

This completes the invalid and anomalous value assessment and establishes the feature-retention decisions for subsequent data cleaning.

In [22]:
# Final feature-retention decision based on dataset-level missingness

high_missing_columns = [
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward"
]

retained_columns = [
    column
    for column in dataset_missing_summary.keys()
    if column not in high_missing_columns
]

feature_retention_analysis = pd.DataFrame({
    "Column": high_missing_columns,
    "Decision": "Remove",
    "Reason": "High missing-value proportion"
})

feature_retention_analysis

,Column,Decision,Reason
0,Init_Win_bytes_forward,Remove,High missing-value proportion
1,Init_Win_bytes_backward,Remove,High missing-value proportion


## 2.5 Missing Value Treatment and Dataset Cleaning

The data-quality assessment identified two features, `Init_Win_bytes_forward` and `Init_Win_bytes_backward`, with substantial missing-value proportions. These features are excluded from the modelling dataset to avoid extensive imputation.

The remaining numerical features contain only a small proportion of missing values. These missing observations are retained as `NaN` during the cleaning process and will be imputed using the median of the corresponding feature.

The raw dataset files are not modified. A separate cleaned dataset is constructed for subsequent preprocessing and feature engineering.

In [23]:
# Apply the final feature-retention decisions
# and prepare the cleaned dataset

cleaned_dataframes = []

for file in csv_files:

    file_path = os.path.join(DATA_PATH, file)

    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    # Remove features with excessive missing values
    df = df.drop(
        columns=high_missing_columns,
        errors="ignore"
    )

    # Replace infinite values with NaN
    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Replace remaining negative values with NaN
    for column in negative_columns:

        if column in df.columns:
            df.loc[df[column] < 0, column] = np.nan

    cleaned_dataframes.append(df)

    del df

cleaned_dataset = pd.concat(
    cleaned_dataframes,
    ignore_index=True
)

del cleaned_dataframes

cleaned_dataset.shape

(2830743, 77)

### 2.5.1 Remaining Missing Value Assessment

After removing features with excessive missingness and converting invalid observations to `NaN`, the consolidated dataset is assessed for remaining missing values. This verification determines the extent of missing data that requires imputation.

In [24]:
# Assess remaining missing values in the consolidated cleaned dataset

remaining_missing = (
    cleaned_dataset.isna()
    .sum()
    .sort_values(ascending=False)
)

remaining_missing_df = pd.DataFrame({
    "Column": remaining_missing.index,
    "Missing Values": remaining_missing.values,
    "Missing Percentage": (
        remaining_missing.values
        / len(cleaned_dataset)
        * 100
    )
})

remaining_missing_df = (
    remaining_missing_df[
        remaining_missing_df["Missing Values"] > 0
    ]
    .reset_index(drop=True)
)

remaining_missing_df

,Column,Missing Values,Missing Percentage
0,Flow Packets/s,2982,0.105343
1,Flow Bytes/s,2952,0.104284
2,Flow IAT Min,2891,0.102129
3,Flow IAT Mean,115,0.004063
4,Flow Duration,115,0.004063
5,Flow IAT Max,115,0.004063
6,Fwd Header Length.1,35,0.001236
7,Fwd Header Length,35,0.001236
8,min_seg_size_forward,35,0.001236
9,Bwd Header Length,22,0.000777


### 2.5.2 Missing Value Imputation

The remaining missing values are limited to a small number of numerical network-flow features. Since the missing proportions are low, the affected observations are retained rather than removed.

Median imputation is applied to the remaining numerical features because network-flow features can contain highly skewed distributions and extreme observations. The median provides a robust estimate of the central tendency without being strongly influenced by extreme values.

The imputation is performed using the consolidated cleaned dataset, and the number of remaining missing values is verified after treatment.

In [25]:
# Impute remaining numerical missing values using the median

imputation_analysis = []

for column in remaining_missing_df["Column"]:

    missing_before = cleaned_dataset[column].isna().sum()

    median_value = cleaned_dataset[column].median()

    cleaned_dataset[column] = cleaned_dataset[column].fillna(
        median_value
    )

    missing_after = cleaned_dataset[column].isna().sum()

    imputation_analysis.append({
        "Column": column,
        "Missing Values Before": missing_before,
        "Median Used": median_value,
        "Missing Values After": missing_after
    })

imputation_analysis_df = pd.DataFrame(
    imputation_analysis
)

imputation_analysis_df

,Column,Missing Values Before,Median Used,Missing Values After
0,Flow Packets/s,2982,109.525598,0
1,Flow Bytes/s,2952,4587.155963,0
2,Flow IAT Min,2891,4.000000,0
3,Flow IAT Mean,115,11443.662280,0
4,Flow Duration,115,31317.000000,0
5,Flow IAT Max,115,30865.000000,0
6,Fwd Header Length.1,35,64.000000,0
7,Fwd Header Length,35,64.000000,0
8,min_seg_size_forward,35,24.000000,0
9,Bwd Header Length,22,40.000000,0


In [26]:
# Verify that no missing values remain

total_missing_after_imputation = cleaned_dataset.isna().sum().sum()

total_missing_after_imputation

np.int64(0)

### 2.5.4 Verification and Conclusion

After feature-level removal and median imputation, no missing values remain in the cleaned dataset.

The two features with high missing-value proportions, `Init_Win_bytes_forward` and `Init_Win_bytes_backward`, were removed from the modelling dataset. The remaining missing values were limited to a small proportion of the data and were imputed using feature-wise median values.

The final verification confirms that the cleaned dataset contains zero missing values. The dataset is therefore ready for subsequent data standardization and preparation steps.

### 2.6 Data Type and Schema Standardization

After handling invalid values and missing data, the cleaned dataset is examined for consistency in column names, data types, and overall schema.

The objective is to ensure that numerical network-flow features are represented using appropriate numerical data types and that the target label is stored consistently.

This step prepares the dataset for subsequent exploratory analysis, feature engineering, and model development.

In [27]:
# Examine the cleaned dataset schema and data types

schema_analysis = pd.DataFrame({
    "Column": cleaned_dataset.columns,
    "Data Type": cleaned_dataset.dtypes.astype(str).values,
    "Missing Values": cleaned_dataset.isna().sum().values,
    "Unique Values": cleaned_dataset.nunique().values
})

schema_analysis

,Column,Data Type,Missing Values,Unique Values
0,Destination Port,int64,0,53805
1,Flow Duration,float64,0,1050894
2,Total Fwd Packets,int64,0,1432
3,Total Backward Packets,int64,0,1747
4,Total Length of Fwd Packets,int64,0,17928
...,...,...,...,...
72,Idle Mean,float64,0,222016
73,Idle Std,float64,0,197616
74,Idle Max,int64,0,149737
75,Idle Min,int64,0,223888


In [28]:
# Check for columns containing identical values

duplicate_columns = []

columns = cleaned_dataset.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):

        if cleaned_dataset[columns[i]].equals(
            cleaned_dataset[columns[j]]
        ):
            duplicate_columns.append({
                "Column 1": columns[i],
                "Column 2": columns[j]
            })

duplicate_columns_df = pd.DataFrame(
    duplicate_columns
)

duplicate_columns_df

,Column 1,Column 2
0,Total Fwd Packets,Subflow Fwd Packets
1,Total Backward Packets,Subflow Bwd Packets
2,Fwd PSH Flags,SYN Flag Count
3,Bwd PSH Flags,Bwd URG Flags
4,Bwd PSH Flags,Fwd Avg Bytes/Bulk
5,Bwd PSH Flags,Fwd Avg Packets/Bulk
6,Bwd PSH Flags,Fwd Avg Bulk Rate
7,Bwd PSH Flags,Bwd Avg Bytes/Bulk
8,Bwd PSH Flags,Bwd Avg Packets/Bulk
9,Bwd PSH Flags,Bwd Avg Bulk Rate


### 2.6.1 Redundant and Constant Feature Removal

The schema analysis identified several features that either contain identical information or have no variation across the consolidated dataset.

Exact duplicate representations of the same network-flow measurement are reduced to a single feature to avoid unnecessary redundancy. Features with only one unique value are also removed because constant features provide no discriminative information for machine-learning models.

The feature-retention decisions are made conservatively to preserve meaningful network-flow information while reducing redundant and non-informative columns.

In [29]:
# Identify constant features

constant_columns = [
    column
    for column in cleaned_dataset.columns
    if cleaned_dataset[column].nunique() <= 1
]

constant_columns

['Bwd PSH Flags',
 'Bwd URG Flags',
 'Fwd Avg Bytes/Bulk',
 'Fwd Avg Packets/Bulk',
 'Fwd Avg Bulk Rate',
 'Bwd Avg Bytes/Bulk',
 'Bwd Avg Packets/Bulk',
 'Bwd Avg Bulk Rate']

In [30]:
# Remove constant and clearly redundant features

redundant_columns = [
    "Subflow Fwd Packets",
    "Subflow Bwd Packets",
    "Fwd Header Length.1"
]

columns_to_remove = constant_columns + redundant_columns

cleaned_dataset = cleaned_dataset.drop(
    columns=columns_to_remove
)

columns_removed_df = pd.DataFrame({
    "Removed Column": columns_to_remove,
    "Reason": (
        ["Constant feature"] * len(constant_columns)
        + ["Redundant duplicate representation"] * len(redundant_columns)
    )
})

columns_removed_df

,Removed Column,Reason
0,Bwd PSH Flags,Constant feature
1,Bwd URG Flags,Constant feature
2,Fwd Avg Bytes/Bulk,Constant feature
3,Fwd Avg Packets/Bulk,Constant feature
4,Fwd Avg Bulk Rate,Constant feature
5,Bwd Avg Bytes/Bulk,Constant feature
6,Bwd Avg Packets/Bulk,Constant feature
7,Bwd Avg Bulk Rate,Constant feature
8,Subflow Fwd Packets,Redundant duplicate representation
9,Subflow Bwd Packets,Redundant duplicate representation


In [31]:
# Verify the cleaned dataset dimensions after schema cleanup

cleaned_dataset.shape

(2830743, 66)

### 2.6.2 Schema Standardization Conclusion

Schema inspection confirmed that the cleaned dataset contains consistent numerical and categorical representations, with no remaining missing values.

Constant features and clearly redundant duplicate representations were removed to reduce unnecessary dimensionality while preserving meaningful network-flow information.

The dataset has been reduced from 77 to 66 columns while retaining all 2,830,743 observations. The resulting dataset is now structurally consistent and ready for the next preprocessing stage.

### 2.7.1 Label Distribution Analysis

The target label distribution is analyzed to understand the composition of the cleaned CICIDS-2017 dataset. The analysis identifies the number and proportion of benign and attack categories and provides an initial assessment of class representation before model development.

In [32]:
# Analyze target label distribution

label_distribution = (
    cleaned_dataset["Label"]
    .value_counts()
    .rename_axis("Label")
    .reset_index(name="Count")
)

label_distribution["Percentage"] = (
    label_distribution["Count"]
    / len(cleaned_dataset)
    * 100
)

label_distribution

,Label,Count,Percentage
0,BENIGN,2273097,80.300366
1,DoS Hulk,231073,8.162981
2,PortScan,158930,5.614427
3,DDoS,128027,4.522735
4,DoS GoldenEye,10293,0.363615
5,FTP-Patator,7938,0.280421
6,SSH-Patator,5897,0.208320
7,DoS slowloris,5796,0.204752
8,DoS Slowhttptest,5499,0.194260
9,Bot,1966,0.069452


### 2.7.2 Label Imbalance Assessment

The label distribution shows substantial class imbalance in the cleaned dataset. The BENIGN class represents the majority of observations, while several attack categories contain relatively few samples.

This imbalance is important for subsequent model development because a model trained directly on the original distribution may become biased toward majority classes and may perform poorly on rare attack categories.

The class distribution is therefore documented before deciding on an appropriate target representation and class-balancing strategy.

In [33]:
# Assess class imbalance

imbalance_analysis = label_distribution.copy()

majority_count = imbalance_analysis["Count"].max()
minority_count = imbalance_analysis["Count"].min()

imbalance_ratio = majority_count / minority_count

imbalance_analysis["Relative to Majority (%)"] = (
    imbalance_analysis["Count"] / majority_count * 100
)

print(f"Number of classes: {len(imbalance_analysis)}")
print(f"Majority class: {imbalance_analysis.loc[0, 'Label']}")
print(f"Majority class count: {majority_count:,}")
print(f"Minority class count: {minority_count:,}")
print(f"Majority-to-minority ratio: {imbalance_ratio:,.2f}:1")

imbalance_analysis

Number of classes: 15
Majority class: BENIGN
Majority class count: 2,273,097
Minority class count: 11
Majority-to-minority ratio: 206,645.18:1


,Label,Count,Percentage,Relative to Majority (%)
0,BENIGN,2273097,80.300366,100.000000
1,DoS Hulk,231073,8.162981,10.165558
2,PortScan,158930,5.614427,6.991783
3,DDoS,128027,4.522735,5.632272
4,DoS GoldenEye,10293,0.363615,0.452818
5,FTP-Patator,7938,0.280421,0.349215
6,SSH-Patator,5897,0.208320,0.259426
7,DoS slowloris,5796,0.204752,0.254983
8,DoS Slowhttptest,5499,0.194260,0.241917
9,Bot,1966,0.069452,0.086490


### 2.7.3 Target Label Validation

The target label column is validated to ensure that it contains no missing values, unexpected representations, or unintended duplicate label categories. Particular attention is given to the attack labels containing non-standard characters to ensure that encoding-related representations do not create separate target classes.

This validation ensures that the target variable is consistent before numerical encoding and model development.

In [34]:
# Validate target labels

label_validation = {
    "Missing Labels": cleaned_dataset["Label"].isna().sum(),
    "Unique Labels": cleaned_dataset["Label"].nunique(),
    "Empty Labels": (cleaned_dataset["Label"].astype(str).str.strip() == "").sum(),
    "Non-String Labels": (~cleaned_dataset["Label"].apply(lambda x: isinstance(x, str))).sum()
}

label_validation_df = pd.DataFrame(
    list(label_validation.items()),
    columns=["Check", "Count"]
)

label_validation_df

,Check,Count
0,Missing Labels,0
1,Unique Labels,15
2,Empty Labels,0
3,Non-String Labels,0


In [35]:
# Display exact target label representations

sorted(cleaned_dataset["Label"].unique())

['BENIGN',
 'Bot',
 'DDoS',
 'DoS GoldenEye',
 'DoS Hulk',
 'DoS Slowhttptest',
 'DoS slowloris',
 'FTP-Patator',
 'Heartbleed',
 'Infiltration',
 'PortScan',
 'SSH-Patator',
 'Web Attack � Brute Force',
 'Web Attack � Sql Injection',
 'Web Attack � XSS']

In [36]:
# Inspect the exact Unicode representation of Web Attack labels

web_attack_labels = [
    label
    for label in cleaned_dataset["Label"].unique()
    if "Web Attack" in label
]

for label in web_attack_labels:
    print(repr(label))
    print([hex(ord(char)) for char in label if char == "�"])
    print()

'Web Attack � Brute Force'
['0xfffd']

'Web Attack � XSS'
['0xfffd']

'Web Attack � Sql Injection'
['0xfffd']



In [37]:
# Repair corrupted Unicode replacement characters in target labels

label_replacements = {
    "Web Attack � Brute Force": "Web Attack - Brute Force",
    "Web Attack � XSS": "Web Attack - XSS",
    "Web Attack � Sql Injection": "Web Attack - Sql Injection"
}

cleaned_dataset["Label"] = cleaned_dataset["Label"].replace(label_replacements)

# Verify the corrected labels
sorted(cleaned_dataset["Label"].unique())

['BENIGN',
 'Bot',
 'DDoS',
 'DoS GoldenEye',
 'DoS Hulk',
 'DoS Slowhttptest',
 'DoS slowloris',
 'FTP-Patator',
 'Heartbleed',
 'Infiltration',
 'PortScan',
 'SSH-Patator',
 'Web Attack - Brute Force',
 'Web Attack - Sql Injection',
 'Web Attack - XSS']

In [38]:
# Verify target labels after correction

print("Unique labels:", cleaned_dataset["Label"].nunique())
print("Missing labels:", cleaned_dataset["Label"].isna().sum())
print("Replacement characters remaining:",
      cleaned_dataset["Label"].astype(str).str.contains("�", regex=False).sum())

Unique labels: 15
Missing labels: 0
Replacement characters remaining: 0


### 2.7.3 Target Label Validation Conclusion

Target validation confirmed that the Label column contains 15 valid classes with no missing, empty, or non-string labels.

Three Web Attack labels were found to contain the Unicode replacement character (U+FFFD), indicating an encoding-related representation issue. The affected labels were normalized by replacing the corrupted separator with a standard hyphen while preserving their original attack-category meaning.

The target column is therefore consistent and ready for numerical encoding.

### 2.7.4 Target Encoding

The validated categorical target labels are converted into numerical class identifiers using label encoding. Each unique network-traffic category is assigned a unique integer while preserving the original class relationships through a separate mapping table.

The original textual labels are retained for interpretation and reporting.

In [39]:
from sklearn.preprocessing import LabelEncoder

# Encode target labels

label_encoder = LabelEncoder()

cleaned_dataset["Target"] = label_encoder.fit_transform(
    cleaned_dataset["Label"]
)

# Create label-to-encoded-value mapping

target_mapping = pd.DataFrame({
    "Label": label_encoder.classes_,
    "Encoded Value": range(len(label_encoder.classes_))
})

target_mapping

,Label,Encoded Value
0,BENIGN,0
1,Bot,1
2,DDoS,2
3,DoS GoldenEye,3
4,DoS Hulk,4
5,DoS Slowhttptest,5
6,DoS slowloris,6
7,FTP-Patator,7
8,Heartbleed,8
9,Infiltration,9


In [40]:
# Verify target encoding

target_validation = {
    "Total Samples": len(cleaned_dataset),
    "Unique Original Labels": cleaned_dataset["Label"].nunique(),
    "Unique Encoded Classes": cleaned_dataset["Target"].nunique(),
    "Missing Target Values": cleaned_dataset["Target"].isna().sum()
}

target_validation

{'Total Samples': 2830743,
 'Unique Original Labels': 15,
 'Unique Encoded Classes': 15,
 'Missing Target Values': np.int64(0)}

### 2.7 Label Analysis and Target Preparation Conclusion

The target variable was analyzed to determine the distribution and severity of class imbalance across the 15 traffic categories. The analysis confirmed that the dataset is highly imbalanced, with BENIGN representing the majority class and several attack categories occurring in very small numbers.

The target labels were validated for missing, empty, and non-string values, and encoding inconsistencies in the Web Attack labels were corrected. The categorical labels were then encoded into numerical target values ranging from 0 to 14 while retaining the original Label column for interpretability.

Final validation confirmed 2,830,743 observations, 15 original labels, 15 encoded target classes, and zero missing target values. The target variable is therefore prepared for subsequent preprocessing and model-development stages.

In [41]:
# ============================================================
# 2.8 Final Dataset Validation & Model-Readiness
# ============================================================

# Final shape
final_shape = cleaned_dataset.shape

print("Final Dataset Shape:", final_shape)
print("Total Samples:", final_shape[0])
print("Total Columns:", final_shape[1])

Final Dataset Shape: (2830743, 67)
Total Samples: 2830743
Total Columns: 67


In [42]:
# ============================================================
# 2.8.1 Feature-Target Separation
# ============================================================

# Define target
target_column = "Target"

# Define original label column retained for interpretation
label_column = "Label"

# Separate features and target
X = cleaned_dataset.drop(columns=[target_column, label_column])
y = cleaned_dataset[target_column]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)
print("Number of Features:", X.shape[1])
print("Number of Target Classes:", y.nunique())

Feature Matrix Shape: (2830743, 65)
Target Shape: (2830743,)
Number of Features: 65
Number of Target Classes: 15


In [43]:
# ============================================================
# 2.8.2 Final Data Integrity Check
# ============================================================

final_validation = {
    "Missing Feature Values": int(X.isna().sum().sum()),
    "Missing Target Values": int(y.isna().sum()),
    "Infinite Feature Values": int(np.isinf(X.select_dtypes(include=np.number)).sum().sum()),
    "Feature Rows": X.shape[0],
    "Target Rows": y.shape[0],
    "Feature Count": X.shape[1],
    "Target Classes": y.nunique()
}

final_validation

{'Missing Feature Values': 0,
 'Missing Target Values': 0,
 'Infinite Feature Values': 0,
 'Feature Rows': 2830743,
 'Target Rows': 2830743,
 'Feature Count': 65,
 'Target Classes': 15}

In [44]:
# ============================================================
# 2.8.3 Final Dataset Summary
# ============================================================

final_dataset_summary = {
    "Total Samples": X.shape[0],
    "Model Features": X.shape[1],
    "Target Classes": y.nunique(),
    "Missing Feature Values": int(X.isna().sum().sum()),
    "Missing Target Values": int(y.isna().sum()),
    "Infinite Feature Values": int(
        np.isinf(X.select_dtypes(include=np.number)).sum().sum()
    ),
    "Original Label Column": "Label",
    "Encoded Target Column": "Target"
}

final_dataset_summary

{'Total Samples': 2830743,
 'Model Features': 65,
 'Target Classes': 15,
 'Missing Feature Values': 0,
 'Missing Target Values': 0,
 'Infinite Feature Values': 0,
 'Original Label Column': 'Label',
 'Encoded Target Column': 'Target'}

### 2.8 Final Dataset Validation & Model-Readiness Conclusion

Final validation confirmed that the cleaned dataset contains 2,830,743 observations and 65 model input features, with 15 target classes. All feature and target missing-value checks returned zero, and no infinite numerical values remain.

The original categorical Label column has been retained for interpretability, while the encoded Target column is prepared for model training. Feature-target separation was also verified, with the feature matrix and target vector containing the same number of observations.

The dataset is therefore structurally consistent, numerically valid, and ready for the subsequent feature engineering, feature selection, data splitting, and model development stages.

## 2.9 Persist Cleaned Dataset

The validated and model-ready dataset is persisted to the project's processed-data directory so that subsequent notebooks do not depend on the active Colab runtime.

The dataset is stored in Parquet format because it preserves the tabular schema and data types while providing efficient storage and loading compared with CSV.

The persisted dataset will serve as the input artifact for subsequent preprocessing, feature engineering, model development, and evaluation stages.

In [45]:
# ============================================================
# 2.9 Persist Cleaned Dataset
# ============================================================

# Define processed data directory
PROCESSED_PATH = os.path.join(PROJECT_PATH, "data", "processed")

# Create directory if it does not already exist
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Output path for the cleaned CICIDS-2017 dataset
CLEANED_DATASET_PATH = os.path.join(
    PROCESSED_PATH,
    "CICIDS2017_cleaned.parquet"
)

# Persist the validated cleaned dataset
cleaned_dataset.to_parquet(
    CLEANED_DATASET_PATH,
    index=False
)

print("Cleaned dataset successfully persisted.")
print("Path:", CLEANED_DATASET_PATH)
print("Shape:", cleaned_dataset.shape)

Cleaned dataset successfully persisted.
Path: /content/drive/MyDrive/AI-Network-Intrusion-Detection-System/data/processed/CICIDS2017_cleaned.parquet
Shape: (2830743, 67)


### 2.9.1 Persisted Dataset Verification

The saved Parquet file is reloaded to verify that the persisted artifact retains the expected shape, schema, target column, and absence of missing values.

In [46]:
# ============================================================
# Verify persisted dataset
# ============================================================

persisted_dataset = pd.read_parquet(CLEANED_DATASET_PATH)

print("Persisted Dataset Shape:", persisted_dataset.shape)
print("Total Samples:", len(persisted_dataset))
print("Total Features:", persisted_dataset.shape[1])

print("\nMissing Values:", persisted_dataset.isna().sum().sum())
print("Infinite Values:", np.isinf(
    persisted_dataset.select_dtypes(include=np.number)
).sum().sum())

print("\nTarget Column Present:", "Target" in persisted_dataset.columns)
print("Unique Target Classes:", persisted_dataset["Target"].nunique())

Persisted Dataset Shape: (2830743, 67)
Total Samples: 2830743
Total Features: 67

Missing Values: 0
Infinite Values: 0

Target Column Present: True
Unique Target Classes: 15


### 2.9.2 Persistence Conclusion

The final cleaned CICIDS-2017 dataset was successfully persisted in Parquet format under the project's processed-data directory. The persisted artifact was reloaded and verified to retain the expected dataset shape, feature schema, target column, and data integrity.

The saved dataset can now be reused independently by subsequent notebooks without rerunning the complete raw-data preprocessing workflow.

## 2.10 Reproducible Dataset Recovery

This section provides a lightweight recovery mechanism for future Colab sessions.

The final processed dataset is loaded from the persisted Parquet artifact rather than relying on variables remaining in the current runtime. This allows subsequent work to continue even after a Colab runtime disconnects.

The raw-data preprocessing and exploratory analysis performed earlier in this notebook do not need to be rerun when the persisted artifact is already available.

In [ ]:
# ============================================================
# 2.10 Reproducible Dataset Recovery
# ============================================================

from google.colab import drive
import os
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

# Define project path
PROJECT_PATH = "/content/drive/MyDrive/AI-Network-Intrusion-Detection-System"

# Define persisted dataset path
CLEANED_DATASET_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed",
    "CICIDS2017_cleaned.parquet"
)

# Verify that the persisted dataset exists
if not os.path.exists(CLEANED_DATASET_PATH):
    raise FileNotFoundError(
        f"Persisted cleaned dataset not found:\n{CLEANED_DATASET_PATH}"
    )

# Load the final cleaned dataset
cleaned_dataset = pd.read_parquet(CLEANED_DATASET_PATH)

# Basic validation
print("Cleaned CICIDS-2017 dataset loaded successfully.")
print("Shape:", cleaned_dataset.shape)
print("Missing values:", cleaned_dataset.isna().sum().sum())
print("Target classes:", cleaned_dataset["Target"].nunique())